Mount Drive and Configure Paths

In [ ]:
# ============================================================
# NOTEBOOK 02 — F4Z FEATURE SCREENING
# STEP 1: PROJECT SETUP
# ============================================================

from google.colab import drive

import os
import json
import gc

import numpy as np
import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

# Mount Google Drive
drive.mount("/content/drive")

# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/ScaleWave"

DATA_DIR = os.path.join(PROJECT_DIR, "Data")

RESULTS_DIR = os.path.join(PROJECT_DIR, "Results")

SCREENING_DIR = os.path.join(
    RESULTS_DIR,
    "02_F4Z_Feature_Screening"
)

os.makedirs(SCREENING_DIR, exist_ok=True)

# ------------------------------------------------------------
# Dataset paths
# ------------------------------------------------------------

normal_file = os.path.join(
    DATA_DIR,
    "BTCUSDT_signed_return_between_minus500_and_plus500_20261027_101854_1aefb791.parquet"
)

extreme_file = os.path.join(
    DATA_DIR,
    "BTCUSDT_signed_return_above_500_or_below_minus500_20261027_093028_e1453352.parquet"
)

# ------------------------------------------------------------
# Verify file accessibility
# ------------------------------------------------------------

required_files = {
    "Non-extreme dataset": normal_file,
    "Extreme dataset": extreme_file,
    "Research configuration": os.path.join(
        RESULTS_DIR, "research_config.json"
    ),
    "Feature coverage": os.path.join(
        RESULTS_DIR, "feature_coverage.csv"
    )
}

print("PROJECT FILE CHECK")
print("=" * 60)

for name, path in required_files.items():

    exists = os.path.isfile(path)

    print(f"{name}: {'FOUND' if exists else 'MISSING'}")

    if not exists:
        print("  Path:", path)

print("\nScreening output directory:")
print(SCREENING_DIR)

Load Research Results

In [ ]:
# ============================================================
# STEP 2: LOAD SAVED INSPECTION RESULTS
# ============================================================

# ------------------------------------------------------------
# Load research configuration
# ------------------------------------------------------------

config_path = os.path.join(
    RESULTS_DIR,
    "research_config.json"
)

with open(config_path, "r") as f:
    research_config = json.load(f)

# ------------------------------------------------------------
# Load saved CSV results
# ------------------------------------------------------------

result_files = {
    "feature_coverage": "feature_coverage.csv",
    "coverage_comparison": "coverage_comparison.csv",
    "invalid_reasons": "invalid_reason_summary.csv",
    "return_distribution": "return_distribution.csv",
    "research_periods": "research_period_summary.csv",
    "positive_episodes": "positive_event_episodes.csv"
}

saved_results = {}

for name, filename in result_files.items():

    path = os.path.join(RESULTS_DIR, filename)

    if os.path.isfile(path):

        saved_results[name] = pd.read_csv(path)

        print(
            f"{name}: "
            f"{len(saved_results[name]):,} rows loaded"
        )

    else:
        print(f"{name}: FILE MISSING")

# ------------------------------------------------------------
# Extract key settings
# ------------------------------------------------------------

EVENT_THRESHOLD = research_config["event_threshold_bps"]

DISCOVERY_END = pd.Timestamp(
    research_config["discovery_end_exclusive"]
)

VALIDATION_START = pd.Timestamp(
    research_config["validation_start_inclusive"]
)

print("\nRESEARCH CONFIGURATION")
print("=" * 60)

print("Symbol:", research_config["symbol"])
print("Target:", research_config["target"])
print("Horizon:", research_config["horizon_minutes"], "minutes")

print("Positive event threshold:", EVENT_THRESHOLD, "bps")

print("Discovery end:", DISCOVERY_END)
print("Validation start:", VALIDATION_START)

# ------------------------------------------------------------
# Display preliminary feature coverage
# ------------------------------------------------------------

if "coverage_comparison" in saved_results:

    coverage_df = saved_results["coverage_comparison"]

    print("\nFEATURE COVERAGE DATA")

    print("Total features:", len(coverage_df))

    display(coverage_df.head(5))

Create Discovery-Period Target

In [ ]:
# ============================================================
# STEP 3: PREPARE DISCOVERY-PERIOD TARGET
# ============================================================

TARGET_COLUMNS = [
    "decision_time",
    "signed_return"
]

ROW_BATCH_SIZE = 65536

target_parts = []

# ------------------------------------------------------------
# Process both datasets
# ------------------------------------------------------------

for dataset_name, file_path in [
    ("NON_EXTREME", normal_file),
    ("EXTREME", extreme_file)
]:

    print(f"\nProcessing {dataset_name}...")

    pf = pq.ParquetFile(file_path)

    row_offset = 0

    dataset_rows = 0
    discovery_rows = 0

    # Read only target-related columns
    for batch in pf.iter_batches(
        batch_size=ROW_BATCH_SIZE,
        columns=TARGET_COLUMNS,
        use_threads=True
    ):

        batch_df = batch.to_pandas()

        batch_size = len(batch_df)

        # Preserve original Parquet row positions
        batch_df["source_row"] = np.arange(
            row_offset,
            row_offset + batch_size,
            dtype=np.int64
        )

        row_offset += batch_size
        dataset_rows += batch_size

        # Ensure UTC timestamps
        batch_df["decision_time"] = pd.to_datetime(
            batch_df["decision_time"],
            utc=True
        )

        # Keep discovery observations only
        batch_df = batch_df.loc[
            batch_df["decision_time"] < DISCOVERY_END
        ].copy()

        if batch_df.empty:
            continue

        # Create binary event target
        batch_df["event"] = (
            batch_df["signed_return"] > EVENT_THRESHOLD
        ).astype(np.uint8)

        # Store source dataset identity
        batch_df["source_dataset"] = dataset_name

        discovery_rows += len(batch_df)

        target_parts.append(batch_df)

    print("Total rows:", f"{dataset_rows:,}")

    print("Discovery rows:", f"{discovery_rows:,}")

    assert row_offset == pf.metadata.num_rows

    del pf
    gc.collect()

# ------------------------------------------------------------
# Combine discovery targets
# ------------------------------------------------------------

discovery_target = pd.concat(
    target_parts,
    ignore_index=True
)

del target_parts
gc.collect()

# Sort chronologically for research summaries
discovery_target = (
    discovery_target
    .sort_values("decision_time")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Calculate event statistics
# ------------------------------------------------------------

total_discovery = len(discovery_target)

positive_events = int(
    discovery_target["event"].sum()
)

non_events = total_discovery - positive_events

base_rate = positive_events / total_discovery

# ------------------------------------------------------------
# Verify against Notebook 01
# ------------------------------------------------------------

assert total_discovery == 1_150_558, (
    "Discovery row count differs from Notebook 01."
)

assert positive_events == 95_773, (
    "Positive-event count differs from Notebook 01."
)

assert not discovery_target["signed_return"].isna().any()

assert discovery_target["decision_time"].lt(
    DISCOVERY_END
).all()

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\nDISCOVERY TARGET SUMMARY")
print("=" * 60)

print("Discovery observations:", f"{total_discovery:,}")

print("Positive events:", f"{positive_events:,}")

print("Non-events:", f"{non_events:,}")

print(f"Positive-event base rate: {base_rate:.4%}")

print("\nDiscovery date range:")

print("Start:", discovery_target["decision_time"].min())

print("End:", discovery_target["decision_time"].max())

print("\nEVENT DISTRIBUTION")

display(
    discovery_target["event"]
    .value_counts()
    .rename_axis("Event")
    .reset_index(name="Rows")
)

print("\nTARGET SAMPLE")

display(
    discovery_target[
        [
            "decision_time",
            "signed_return",
            "event",
            "source_dataset",
            "source_row"
        ]
    ].head(10)
)

Save the Prepared Discovery Target

Save Target

In [ ]:
# ============================================================
# STEP 4: SAVE DISCOVERY TARGET
# ============================================================

target_output_path = os.path.join(
    SCREENING_DIR,
    "discovery_target.parquet"
)

# Convert the small target dataframe to Arrow
target_table = pa.Table.from_pandas(
    discovery_target,
    preserve_index=False
)

# Save using compression
pq.write_table(
    target_table,
    target_output_path,
    compression="zstd"
)

print("DISCOVERY TARGET SAVED")
print("=" * 60)

print("File:", target_output_path)

print(
    "Rows:",
    f"{target_table.num_rows:,}"
)

print(
    "Size:",
    f"{os.path.getsize(target_output_path) / (1024**2):.2f} MB"
)

Build the F4Z Feature Catalog

Identify and Organize F4Z Features

In [ ]:
# ============================================================
# STEP 5: BUILD F4Z FEATURE CATALOG
# ============================================================

import re

# Load the existing feature coverage comparison
coverage_df = saved_results["coverage_comparison"].copy()

# ------------------------------------------------------------
# Identify actual F4Z value columns
# ------------------------------------------------------------

f4z_columns = sorted(
    col
    for col in coverage_df["Feature"].dropna().unique()
    if col.startswith("f4z__")
    and col.endswith("__value")
)

print("F4Z value columns identified:", len(f4z_columns))

assert len(f4z_columns) == 522, (
    "Expected 522 F4Z value columns."
)

# ------------------------------------------------------------
# Parse feature names
# ------------------------------------------------------------

def parse_f4z_feature(column):

    parts = column.split("__")

    family = parts[1]

    feature_name = "__".join(parts[2:-2])

    lookback = parts[-2]

    # Common transformation suffixes.
    # These are inferred from naming conventions.
    transforms = [
        "_percentile",
        "_robust_zscore",
        "_zscore",
        "_rz",
    ]

    parent_sensor = feature_name
    transformation = "unknown"

    for suffix in sorted(transforms, key=len, reverse=True):

        if feature_name.endswith(suffix):

            parent_sensor = feature_name[:-len(suffix)]

            transformation = suffix.lstrip("_")

            break

    return {
        "Feature": column,
        "Family_Parsed": family,
        "Parent_Sensor": parent_sensor,
        "Transformation": transformation,
        "Lookback": lookback
    }


feature_catalog = pd.DataFrame(
    [parse_f4z_feature(col) for col in f4z_columns]
)

# ------------------------------------------------------------
# Add coverage statistics from Notebook 01
# ------------------------------------------------------------

feature_catalog = feature_catalog.merge(
    coverage_df[
        [
            "Feature",
            "Extreme_Coverage",
            "Non_Extreme_Coverage"
        ]
    ],
    on="Feature",
    how="left",
    validate="one_to_one"
)

# These two coverage columns refer to the original
# extreme/non-extreme datasets, NOT event/non-event groups.

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nFEATURE CATALOG SUMMARY")
print("=" * 60)

print("Total F4Z features:", len(feature_catalog))

print(
    "Families:",
    feature_catalog["Family_Parsed"].nunique()
)

print(
    "Inferred parent sensors:",
    feature_catalog["Parent_Sensor"].nunique()
)

print("\nFEATURES BY FAMILY")

display(
    feature_catalog.groupby(
        "Family_Parsed"
    ).size().reset_index(name="Feature_Count")
)

print("\nTRANSFORMATION TYPES")

display(
    feature_catalog["Transformation"]
    .value_counts()
    .rename_axis("Transformation")
    .reset_index(name="Count")
)

print("\nFEATURE CATALOG SAMPLE")

display(
    feature_catalog[
        [
            "Feature",
            "Family_Parsed",
            "Parent_Sensor",
            "Transformation",
            "Lookback",
            "Non_Extreme_Coverage",
            "Extreme_Coverage"
        ]
    ].head(15)
)

# ------------------------------------------------------------
# Save catalog
# ------------------------------------------------------------

catalog_path = os.path.join(
    SCREENING_DIR,
    "f4z_feature_catalog.csv"
)

feature_catalog.to_csv(
    catalog_path,
    index=False
)

print("\nFeature catalog saved:")
print(catalog_path)

Inspect Unknown Transformations

In [ ]:
# ============================================================
# STEP 6: INSPECT UNKNOWN F4Z TRANSFORMATIONS
# ============================================================

# Select features not recognized by our initial parser
unknown_features = feature_catalog.loc[
    feature_catalog["Transformation"] == "unknown"
].copy()

print("UNKNOWN TRANSFORMATION ANALYSIS")
print("=" * 60)

print("Unknown features:", len(unknown_features))

# ------------------------------------------------------------
# Distribution by family
# ------------------------------------------------------------

print("\nUNKNOWN FEATURES BY FAMILY")

display(
    unknown_features.groupby("Family_Parsed")
    .size()
    .reset_index(name="Unknown_Count")
    .sort_values("Unknown_Count", ascending=False)
)

# ------------------------------------------------------------
# Identify naming patterns
# ------------------------------------------------------------

unknown_features["Feature_Name"] = (
    unknown_features["Feature"]
    .str.split("__")
    .str[2]
)

print("\nUNRECOGNIZED FEATURE NAME PATTERNS")

display(
    unknown_features["Feature_Name"]
    .value_counts()
    .rename_axis("Feature_Name")
    .reset_index(name="Count")
)

# ------------------------------------------------------------
# Display distinct examples
# ------------------------------------------------------------

# ------------------------------------------------------------
# Display distinct examples — CORRECTED
# ------------------------------------------------------------

print("\nUNKNOWN FEATURE EXAMPLES")

display(
    unknown_features[
        [
            "Feature",
            "Feature_Name",
            "Family_Parsed",
            "Parent_Sensor",
            "Transformation",
            "Lookback"
        ]
    ].drop_duplicates(
        subset=["Feature_Name"]
    ).head(35)
)

print("\nTOTAL DISTINCT UNKNOWN FEATURE NAMES:")

print(
    unknown_features["Feature_Name"].nunique()
)

Correct Feature Catalog

In [ ]:
# ============================================================
# STEP 7: CORRECT F4Z FEATURE CATALOG
# ============================================================

import re

# Recognized lookback formats
LOOKBACK_PATTERN = re.compile(
    r"^\d+(?:m|h|d|w)$"
)

# Recognized transformations
TRANSFORM_SUFFIXES = [
    "_historical_active_rate",
    "_percentile",
    "_robust_zscore",
    "_zscore",
    "_rz"
]

def parse_f4z_feature_v2(column):

    parts = column.split("__")

    if (
        len(parts) < 5
        or parts[0] != "f4z"
        or parts[-1] != "value"
    ):
        raise ValueError(f"Unexpected F4Z column: {column}")

    family = parts[1]

    feature_name = parts[2]

    # Everything between feature name and value
    lookback_parts = parts[3:-1]

    # Validate lookback tokens without discarding them
    if not all(
        LOOKBACK_PATTERN.fullmatch(p)
        for p in lookback_parts
    ):
        raise ValueError(
            f"Unrecognized lookback structure: {column}"
        )

    parent_sensor = feature_name
    transformation = "unknown"

    for suffix in sorted(
        TRANSFORM_SUFFIXES,
        key=len,
        reverse=True
    ):

        if feature_name.endswith(suffix):

            parent_sensor = feature_name[:-len(suffix)]

            transformation = suffix.lstrip("_")

            break

    return {
        "Feature": column,
        "Family": family,
        "Parent_Sensor": parent_sensor,
        "Transformation": transformation,
        "Lookback_Count": len(lookback_parts),
        "Lookback_1": (
            lookback_parts[0]
            if len(lookback_parts) >= 1
            else None
        ),
        "Lookback_2": (
            lookback_parts[1]
            if len(lookback_parts) >= 2
            else None
        )
    }


# ------------------------------------------------------------
# Rebuild feature catalog
# ------------------------------------------------------------

feature_catalog_v2 = pd.DataFrame(
    [
        parse_f4z_feature_v2(column)
        for column in f4z_columns
    ]
)

# Add saved coverage
feature_catalog_v2 = feature_catalog_v2.merge(
    coverage_df[
        [
            "Feature",
            "Extreme_Coverage",
            "Non_Extreme_Coverage"
        ]
    ],
    on="Feature",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# Quality checks
# ------------------------------------------------------------

assert len(feature_catalog_v2) == 522

assert feature_catalog_v2["Feature"].is_unique

print("CORRECTED FEATURE CATALOG")
print("=" * 60)

print("Total features:", len(feature_catalog_v2))

print(
    "Feature families:",
    feature_catalog_v2["Family"].nunique()
)

print(
    "Inferred parent sensors:",
    feature_catalog_v2["Parent_Sensor"].nunique()
)

print("\nTRANSFORMATION DISTRIBUTION")

display(
    feature_catalog_v2["Transformation"]
    .value_counts()
    .rename_axis("Transformation")
    .reset_index(name="Count")
)

print("\nLOOKBACK STRUCTURE")

display(
    feature_catalog_v2["Lookback_Count"]
    .value_counts()
    .sort_index()
    .rename_axis("Number_of_Lookbacks")
    .reset_index(name="Features")
)

print("\nUNRECOGNIZED FEATURES")

unrecognized = feature_catalog_v2.loc[
    feature_catalog_v2["Transformation"] == "unknown"
]

print("Remaining unknown:", len(unrecognized))

display(unrecognized.head(20))

print("\nCORRECTED FEATURE EXAMPLES")

examples = [
    "flow_acceleration_rz",
    "liq_fast_slow_lift_percentile",
    "liq_activity_severity_historical_active_rate"
]

display(
    feature_catalog_v2.loc[
        feature_catalog_v2["Feature"].apply(
            lambda x: any(name in x for name in examples)
        )
    ].head(12)
)

# ------------------------------------------------------------
# Save corrected catalog
# ------------------------------------------------------------

catalog_v2_path = os.path.join(
    SCREENING_DIR,
    "f4z_feature_catalog_v2.csv"
)

feature_catalog_v2.to_csv(
    catalog_v2_path,
    index=False
)

print("\nCorrected feature catalog saved:")
print(catalog_v2_path)

Verify Feature and Validity Column Mapping

Verify Feature and Validity Mapping

In [ ]:
# ============================================================
# STEP 8: VERIFY FEATURE AND VALIDITY COLUMN MAPPING
# ============================================================

import pyarrow.parquet as pq

# Open metadata only
normal_pf = pq.ParquetFile(normal_file)
extreme_pf = pq.ParquetFile(extreme_file)

normal_schema = normal_pf.schema_arrow
extreme_schema = extreme_pf.schema_arrow

normal_columns = set(normal_schema.names)
extreme_columns = set(extreme_schema.names)

print("FEATURE SCHEMA VERIFICATION")
print("=" * 60)

print("Non-extreme columns:", len(normal_columns))
print("Extreme columns:", len(extreme_columns))

# ------------------------------------------------------------
# Verify selected F4Z columns exist
# ------------------------------------------------------------

missing_normal = [
    col for col in f4z_columns
    if col not in normal_columns
]

missing_extreme = [
    col for col in f4z_columns
    if col not in extreme_columns
]

print("\nMissing F4Z columns")

print("Non-extreme:", len(missing_normal))
print("Extreme:", len(missing_extreme))

assert not missing_normal
assert not missing_extreme

# ------------------------------------------------------------
# Identify associated validity columns
# ------------------------------------------------------------

validity_mapping = {}

missing_validity = []

for feature in f4z_columns:

    validity_col = feature.replace(
        "__value",
        "__is_valid"
    )

    if (
        validity_col in normal_columns
        and validity_col in extreme_columns
    ):

        validity_mapping[feature] = validity_col

    else:
        missing_validity.append(feature)

print("\nVALIDITY MAPPING")

print("Features:", len(f4z_columns))

print("Mapped validity columns:", len(validity_mapping))

print("Missing validity columns:", len(missing_validity))

# ------------------------------------------------------------
# Verify target alignment information
# ------------------------------------------------------------

print("\nDISCOVERY TARGET ALIGNMENT")

required_target_columns = [
    "decision_time",
    "signed_return",
    "event",
    "source_dataset",
    "source_row"
]

missing_target = [
    col for col in required_target_columns
    if col not in discovery_target.columns
]

print("Missing target columns:", missing_target)

assert not missing_target

# Ensure source row identifiers are unique within each dataset
assert not discovery_target.duplicated(
    subset=["source_dataset", "source_row"]
).any()

# ------------------------------------------------------------
# Display sample mapping
# ------------------------------------------------------------

mapping_df = pd.DataFrame(
    [
        {
            "Feature": feature,
            "Validity_Column": validity_mapping[feature]
        }
        for feature in validity_mapping
    ]
)

print("\nSAMPLE FEATURE-VALIDITY MAPPING")

display(mapping_df.head(10))

# ------------------------------------------------------------
# Save mapping
# ------------------------------------------------------------

mapping_path = os.path.join(
    SCREENING_DIR,
    "f4z_validity_mapping.csv"
)

mapping_df.to_csv(mapping_path, index=False)

print("\nValidity mapping saved:")
print(mapping_path)

# Release Parquet metadata references
del normal_pf, extreme_pf

Inspect Actual Validity Column Names

Identify Validity Naming Convention

In [ ]:
# ============================================================
# STEP 8A: INSPECT ACTUAL VALIDITY COLUMN NAMES
# ============================================================

import pyarrow.parquet as pq

# Read schema metadata only
pf = pq.ParquetFile(normal_file)

all_columns = pf.schema_arrow.names

print("COLUMN NAMING INVESTIGATION")
print("=" * 60)

# ------------------------------------------------------------
# Identify columns containing validity-related terminology
# ------------------------------------------------------------

validity_keywords = [
    "valid",
    "invalid",
    "status",
    "reason"
]

validity_related = [
    col for col in all_columns
    if any(
        keyword in col.lower()
        for keyword in validity_keywords
    )
]

print("\nValidity-related columns:", len(validity_related))

print("\nFIRST 35 VALIDITY-RELATED COLUMNS")

for col in validity_related[:35]:
    print(col)

# ------------------------------------------------------------
# Inspect every column associated with one known feature
# ------------------------------------------------------------

example_feature = (
    "f4z__basis__basis_change_percentile__12h__value"
)

example_prefix = example_feature.removesuffix("__value")

related_columns = [
    col for col in all_columns
    if col.startswith(example_prefix)
]

print("\nCOLUMNS RELATED TO EXAMPLE FEATURE")
print("=" * 60)

for col in related_columns:
    print(col)

# ------------------------------------------------------------
# Inspect basis-family column names
# ------------------------------------------------------------

basis_columns = [
    col for col in all_columns
    if col.startswith("f4z__basis__")
]

print("\nFIRST 25 BASIS FAMILY COLUMNS")

for col in basis_columns[:25]:
    print(col)

del pf

Create the Correct Mapping

In [ ]:
# ============================================================
# STEP 8B: CORRECT F4Z VALIDITY MAPPING
# ============================================================

import pyarrow as pa
import pyarrow.parquet as pq

# Read schemas only
normal_schema = pq.read_schema(normal_file)
extreme_schema = pq.read_schema(extreme_file)

normal_columns = set(normal_schema.names)
extreme_columns = set(extreme_schema.names)

validity_mapping = {}
missing_validity = []
non_boolean_validity = []

for feature in f4z_columns:

    # Correct suffix
    validity_col = feature.removesuffix("__value") + "__valid"

    # Check both datasets
    if (
        validity_col in normal_columns
        and validity_col in extreme_columns
    ):

        validity_mapping[feature] = validity_col

        # Check data types
        if not (
            pa.types.is_boolean(normal_schema.field(validity_col).type)
            and
            pa.types.is_boolean(extreme_schema.field(validity_col).type)
        ):
            non_boolean_validity.append(validity_col)

    else:
        missing_validity.append(feature)

# ------------------------------------------------------------
# Quality checks
# ------------------------------------------------------------

print("CORRECTED VALIDITY MAPPING")
print("=" * 60)

print("Total features:", len(f4z_columns))
print("Mapped validity columns:", len(validity_mapping))
print("Missing validity columns:", len(missing_validity))
print("Non-Boolean validity columns:", len(non_boolean_validity))

assert len(validity_mapping) == 522
assert len(missing_validity) == 0
assert len(non_boolean_validity) == 0

# ------------------------------------------------------------
# Save corrected mapping
# ------------------------------------------------------------

mapping_df = pd.DataFrame(
    [
        {
            "Feature": feature,
            "Validity_Column": validity_mapping[feature]
        }
        for feature in f4z_columns
    ]
)

mapping_path = os.path.join(
    SCREENING_DIR,
    "f4z_validity_mapping.csv"
)

mapping_df.to_csv(mapping_path, index=False)

print("\nSAMPLE MAPPING")

display(mapping_df.head(8))

print("\nCorrected mapping saved:")
print(mapping_path)

Test a Single Feature

In [ ]:
# ============================================================
# STEP 9A: PILOT NUMERICAL FEATURE SCREENING
# ============================================================

import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

start_time = time.time()

# ------------------------------------------------------------
# Select one feature for technical testing
# ------------------------------------------------------------

TEST_FEATURE = (
    "f4z__basis__basis_change_percentile__5m__value"
)

VALID_COL = validity_mapping[TEST_FEATURE]

# ------------------------------------------------------------
# Prepare discovery target lookup
# ------------------------------------------------------------

target_lookup = {}

for source_name in ["NON_EXTREME", "EXTREME"]:

    subset = discovery_target.loc[
        discovery_target["source_dataset"] == source_name,
        ["source_row", "event"]
    ].sort_values("source_row")

    target_lookup[source_name] = {
        "rows": subset["source_row"].to_numpy(
            dtype=np.int64
        ),
        "events": subset["event"].to_numpy(
            dtype=np.uint8
        )
    }

# ------------------------------------------------------------
# Read feature values in batches
# ------------------------------------------------------------

value_parts = []
event_parts = []

for source_name, file_path in [
    ("NON_EXTREME", normal_file),
    ("EXTREME", extreme_file)
]:

    print(f"\nReading {source_name}...")

    pf = pq.ParquetFile(file_path)

    target_rows = target_lookup[source_name]["rows"]
    target_events = target_lookup[source_name]["events"]

    row_offset = 0

    for batch in pf.iter_batches(
        batch_size=65536,
        columns=[TEST_FEATURE, VALID_COL],
        use_threads=True
    ):

        batch_end = row_offset + batch.num_rows

        # Find discovery target rows in this batch
        left = np.searchsorted(target_rows, row_offset)
        right = np.searchsorted(target_rows, batch_end)

        if right > left:

            source_positions = (
                target_rows[left:right] - row_offset
            )

            # Read Boolean validity
            valid_array = batch.column(1)

            valid_flags = valid_array.to_numpy(
                zero_copy_only=False
            )

            valid_flags = np.asarray(
                valid_flags,
                dtype=bool
            )

            # Null validity flags are not usable
            if valid_array.null_count:
                nonnull_flags = pc.is_valid(
                    valid_array
                ).to_numpy(zero_copy_only=False)

                valid_flags &= nonnull_flags

            # Read only the selected discovery rows
            selected_valid = valid_flags[source_positions]

            if selected_valid.any():

                selected_positions = source_positions[
                    selected_valid
                ]

                # Convert selected decimal values to float64
                selected_values = pc.take(
                    batch.column(0),
                    pa.array(selected_positions)
                )

                selected_values = pc.cast(
                    selected_values,
                    pa.float64(),
                    safe=False
                )

                values = selected_values.to_numpy(
                    zero_copy_only=False
                )

                events = target_events[left:right][
                    selected_valid
                ]

                # Exclude null, NaN, and infinite readings
                finite_mask = np.isfinite(values)

                if finite_mask.any():

                    value_parts.append(
                        values[finite_mask]
                    )

                    event_parts.append(
                        events[finite_mask]
                    )

        row_offset = batch_end

    assert row_offset == pf.metadata.num_rows

    print("Completed.")

# ------------------------------------------------------------
# Combine valid discovery readings
# ------------------------------------------------------------

feature_values = np.concatenate(value_parts)

feature_events = np.concatenate(event_parts)

assert len(feature_values) == len(feature_events)

valid_count = len(feature_values)

event_count = int(feature_events.sum())

valid_event_rate = event_count / valid_count

coverage = valid_count / len(discovery_target)

# ------------------------------------------------------------
# Determine high and low thresholds
# ------------------------------------------------------------

q10, q90 = np.quantile(
    feature_values,
    [0.10, 0.90]
)

# ------------------------------------------------------------
# Evaluate feature conditions
# ------------------------------------------------------------

conditions = {
    "High (>= 90th percentile)": feature_values >= q90,
    "Low (<= 10th percentile)": feature_values <= q10,
    "Positive (> 0)": feature_values > 0,
    "Negative (< 0)": feature_values < 0
}

results = []

for condition_name, condition_mask in conditions.items():

    n = int(condition_mask.sum())

    positive_n = int(
        feature_events[condition_mask].sum()
    )

    if n == 0:
        probability = np.nan
        lift = np.nan
    else:
        probability = positive_n / n
        lift = probability / valid_event_rate

    results.append({
        "Condition": condition_name,
        "Observations": n,
        "Positive_Events": positive_n,
        "Event_Probability": probability,
        "Lift_vs_Valid_Baseline": lift
    })

pilot_results = pd.DataFrame(results)

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("\nPILOT FEATURE SCREENING")
print("=" * 65)

print("Feature:", TEST_FEATURE)

print("\nDISCOVERY VALIDITY SUMMARY")

print("Discovery rows:", f"{len(discovery_target):,}")
print("Valid feature readings:", f"{valid_count:,}")
print(f"Feature coverage: {coverage:.2%}")

print("Positive events among valid readings:", f"{event_count:,}")
print(f"Feature-valid event baseline: {valid_event_rate:.4%}")

print(f"\n10th percentile threshold: {q10:.6f}")
print(f"90th percentile threshold: {q90:.6f}")

print("\nCONDITIONAL EVENT PROBABILITIES")

display(
    pilot_results.style.format({
        "Event_Probability": "{:.4%}",
        "Lift_vs_Valid_Baseline": "{:.3f}x"
    })
)

print(
    "\nExecution time:",
    f"{time.time() - start_time:.2f} seconds"
)

Create a Screening Plan

Build Feature Batches

In [ ]:
# ============================================================
# STEP 9B: PREPARE FULL FEATURE SCREENING
# ============================================================

import math

# Number of features processed together
FEATURE_BATCH_SIZE = 10

# All 522 features
all_screening_features = feature_catalog_v2[
    "Feature"
].tolist()

# ------------------------------------------------------------
# Create batches
# ------------------------------------------------------------

feature_batches = [
    all_screening_features[i:i + FEATURE_BATCH_SIZE]
    for i in range(
        0,
        len(all_screening_features),
        FEATURE_BATCH_SIZE
    )
]

# ------------------------------------------------------------
# Verify batch completeness
# ------------------------------------------------------------

flattened_features = [
    feature
    for batch in feature_batches
    for feature in batch
]

assert len(flattened_features) == 522

assert len(set(flattened_features)) == 522

assert set(flattened_features) == set(f4z_columns)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("FULL F4Z SCREENING PLAN")
print("=" * 60)

print("Total features:", len(all_screening_features))

print("Features per batch:", FEATURE_BATCH_SIZE)

print("Total batches:", len(feature_batches))

print("\nFirst batch:")

for i, feature in enumerate(feature_batches[0], 1):
    print(f"{i:2}. {feature}")

print("\nLast batch size:", len(feature_batches[-1]))

# ------------------------------------------------------------
# Save screening plan
# ------------------------------------------------------------

screening_plan = {
    "total_features": len(all_screening_features),
    "feature_batch_size": FEATURE_BATCH_SIZE,
    "total_batches": len(feature_batches),
    "target": "signed_return",
    "positive_event_condition": "signed_return > 500",
    "period": "discovery_only",
    "high_quantile": 0.90,
    "low_quantile": 0.10
}

plan_path = os.path.join(
    SCREENING_DIR,
    "screening_plan.json"
)

with open(plan_path, "w") as f:
    json.dump(screening_plan, f, indent=4)

print("\nScreening plan saved:")
print(plan_path)

Screen the First 10 Features

In [ ]:
# ============================================================
# STEP 9C: SCREEN FIRST BATCH OF 10 F4Z FEATURES
# ============================================================

import time
import gc
import os

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

start_time = time.time()

# ------------------------------------------------------------
# 1. Select first feature batch
# ------------------------------------------------------------

BATCH_NUMBER = 0

selected_features = feature_batches[BATCH_NUMBER]

print("SCREENING BATCH:", BATCH_NUMBER + 1)
print("Number of features:", len(selected_features))

# ------------------------------------------------------------
# 2. Prepare target lookups
# ------------------------------------------------------------

target_lookup = {}

for source_name in ["NON_EXTREME", "EXTREME"]:

    subset = (
        discovery_target.loc[
            discovery_target["source_dataset"] == source_name,
            ["source_row", "event"]
        ]
        .sort_values("source_row")
    )

    target_lookup[source_name] = {
        "rows": subset["source_row"].to_numpy(
            dtype=np.int64
        ),
        "events": subset["event"].to_numpy(
            dtype=np.uint8
        )
    }

# ------------------------------------------------------------
# 3. Prepare storage for valid feature readings
# ------------------------------------------------------------

feature_storage = {
    feature: {
        "values": [],
        "events": []
    }
    for feature in selected_features
}

# Read only necessary columns
required_columns = []

for feature in selected_features:
    required_columns.append(feature)
    required_columns.append(validity_mapping[feature])

# ------------------------------------------------------------
# 4. Process both Parquet files
# ------------------------------------------------------------

for source_name, file_path in [
    ("NON_EXTREME", normal_file),
    ("EXTREME", extreme_file)
]:

    print(f"\nProcessing {source_name}...")

    pf = pq.ParquetFile(file_path)

    target_rows = target_lookup[source_name]["rows"]
    target_events = target_lookup[source_name]["events"]

    row_offset = 0

    for batch in pf.iter_batches(
        batch_size=32768,
        columns=required_columns,
        use_threads=True
    ):

        batch_end = row_offset + batch.num_rows

        # Locate discovery observations in current batch
        left = np.searchsorted(target_rows, row_offset)
        right = np.searchsorted(target_rows, batch_end)

        if right == left:
            row_offset = batch_end
            continue

        selected_rows = (
            target_rows[left:right] - row_offset
        )

        selected_events = target_events[left:right]

        # ----------------------------------------------------
        # Process each feature independently
        # ----------------------------------------------------

        for feature_index, feature in enumerate(
            selected_features
        ):

            value_column = batch.column(feature_index * 2)

            valid_column = batch.column(
                feature_index * 2 + 1
            )

            # Extract validity flags
            valid_flags = valid_column.to_numpy(
                zero_copy_only=False
            )

            valid_flags = np.asarray(
                valid_flags,
                dtype=bool
            )

            # Exclude null validity flags
            if valid_column.null_count:

                nonnull = pc.is_valid(
                    valid_column
                ).to_numpy(zero_copy_only=False)

                valid_flags &= nonnull

            selected_valid = valid_flags[selected_rows]

            if not selected_valid.any():
                continue

            valid_positions = selected_rows[selected_valid]

            # Retrieve selected numerical values
            values_arrow = pc.take(
                value_column,
                pa.array(valid_positions)
            )

            # Convert decimal/numeric values to float64
            values_arrow = pc.cast(
                values_arrow,
                pa.float64(),
                safe=False
            )

            values = values_arrow.to_numpy(
                zero_copy_only=False
            )

            events = selected_events[selected_valid]

            # Exclude null and non-finite values
            finite = np.isfinite(values)

            if not finite.any():
                continue

            feature_storage[feature]["values"].append(
                values[finite]
            )

            feature_storage[feature]["events"].append(
                events[finite]
            )

        row_offset = batch_end

    assert row_offset == pf.metadata.num_rows

    del pf
    gc.collect()

    print("Completed.")

# ------------------------------------------------------------
# 5. Calculate screening metrics
# ------------------------------------------------------------

screening_results = []

total_discovery = len(discovery_target)

for feature in selected_features:

    value_parts = feature_storage[feature]["values"]
    event_parts = feature_storage[feature]["events"]

    if not value_parts:
        screening_results.append({
            "Feature": feature,
            "Condition": "NO_VALID_READINGS",
            "Valid_Observations": 0,
            "Coverage": 0.0,
            "Valid_Event_Rate": np.nan,
            "Threshold": np.nan,
            "Condition_Observations": 0,
            "Condition_Events": 0,
            "Event_Probability": np.nan,
            "Lift": np.nan
        })
        continue

    values = np.concatenate(value_parts)
    events = np.concatenate(event_parts)

    valid_n = len(values)

    valid_events = int(events.sum())

    valid_event_rate = valid_events / valid_n

    coverage = valid_n / total_discovery

    q10, q90 = np.quantile(
        values,
        [0.10, 0.90]
    )

    conditions = {
        "HIGH_90": (values >= q90, q90),
        "LOW_10": (values <= q10, q10),
        "POSITIVE": (values > 0, 0.0),
        "NEGATIVE": (values < 0, 0.0)
    }

    for condition_name, (mask, threshold) in conditions.items():

        condition_n = int(mask.sum())

        condition_events = int(events[mask].sum())

        if condition_n > 0:

            probability = (
                condition_events / condition_n
            )

            lift = (
                probability / valid_event_rate
                if valid_event_rate > 0
                else np.nan
            )

        else:
            probability = np.nan
            lift = np.nan

        screening_results.append({
            "Feature": feature,
            "Condition": condition_name,
            "Valid_Observations": valid_n,
            "Coverage": coverage,
            "Valid_Event_Rate": valid_event_rate,
            "Threshold": threshold,
            "Condition_Observations": condition_n,
            "Condition_Events": condition_events,
            "Event_Probability": probability,
            "Lift": lift
        })

    del values, events

# ------------------------------------------------------------
# 6. Create results dataframe
# ------------------------------------------------------------

batch_results = pd.DataFrame(screening_results)

# Add feature metadata
batch_results = batch_results.merge(
    feature_catalog_v2[
        [
            "Feature",
            "Family",
            "Parent_Sensor",
            "Transformation",
            "Lookback_1",
            "Lookback_2"
        ]
    ],
    on="Feature",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 7. Display results
# ------------------------------------------------------------

print("\nBATCH SCREENING SUMMARY")
print("=" * 65)

print("Features screened:", len(selected_features))

print("Result rows:", len(batch_results))

print("\nTOP CONDITIONS BY OBSERVED LIFT")

display(
    batch_results.loc[
        batch_results["Condition_Observations"] >= 1000
    ]
    .sort_values("Lift", ascending=False)
    [
        [
            "Feature",
            "Condition",
            "Coverage",
            "Condition_Observations",
            "Condition_Events",
            "Event_Probability",
            "Lift"
        ]
    ]
    .head(15)
)

# ------------------------------------------------------------
# 8. Save batch results
# ------------------------------------------------------------

batch_output_path = os.path.join(
    SCREENING_DIR,
    "screening_batch_01.csv"
)

batch_results.to_csv(
    batch_output_path,
    index=False
)

print("\nRESULTS SAVED:")
print(batch_output_path)

print(
    "\nExecution time:",
    f"{time.time() - start_time:.2f} seconds"
)

# Release stored feature arrays
del feature_storage
gc.collect()

Verify Batch 1 Before Scaling Up

Validate Screening Results

In [ ]:
# ============================================================
# STEP 9D: VALIDATE FIRST SCREENING BATCH
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Load saved batch results
# ------------------------------------------------------------

batch_path = os.path.join(
    SCREENING_DIR,
    "screening_batch_01.csv"
)

results = pd.read_csv(batch_path)

print("BATCH 1 VALIDATION")
print("=" * 60)

print("Result rows:", len(results))
print("Unique features:", results["Feature"].nunique())

# ------------------------------------------------------------
# Identify features without usable readings
# ------------------------------------------------------------

no_valid = results.loc[
    results["Condition"] == "NO_VALID_READINGS",
    ["Feature", "Valid_Observations", "Coverage"]
]

print("\nFEATURES WITHOUT VALID READINGS")

display(no_valid)

# ------------------------------------------------------------
# Check results per feature
# ------------------------------------------------------------

print("\nRESULT ROWS PER FEATURE")

feature_counts = (
    results.groupby("Feature")
    .size()
    .reset_index(name="Result_Rows")
)

display(feature_counts)

# ------------------------------------------------------------
# Verify pilot feature
# ------------------------------------------------------------

pilot_feature = (
    "f4z__basis__basis_change_percentile__5m__value"
)

pilot_comparison = results.loc[
    results["Feature"] == pilot_feature,
    [
        "Condition",
        "Coverage",
        "Event_Probability",
        "Lift"
    ]
]

print("\nPILOT FEATURE CONSISTENCY CHECK")

display(pilot_comparison)

expected_values = {
    "HIGH_90": (0.082849, 0.991),
    "LOW_10": (0.086341, 1.033)
}

for condition, (expected_probability, expected_lift) in expected_values.items():

    row = pilot_comparison.loc[
        pilot_comparison["Condition"] == condition
    ]

    assert len(row) == 1

    actual_probability = row["Event_Probability"].iloc[0]
    actual_lift = row["Lift"].iloc[0]

    assert np.isclose(
        actual_probability,
        expected_probability,
        atol=0.00001
    )

    assert np.isclose(
        actual_lift,
        expected_lift,
        atol=0.002
    )

print("\nPilot consistency: PASSED")

# ------------------------------------------------------------
# Verify feature completeness
# ------------------------------------------------------------

expected_features = set(feature_batches[0])
observed_features = set(results["Feature"])

assert expected_features == observed_features

print("Feature completeness: PASSED")

# Verify the expected number of result rows
assert len(results) == (
    4 * (len(expected_features) - len(no_valid))
    + len(no_valid)
)

print("Result structure: PASSED")

Reusable F4Z Screening Function

In [ ]:
# ============================================================
# STEP 9E: REUSABLE F4Z SCREENING FUNCTION
# ============================================================

import os
import gc
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc


# ------------------------------------------------------------
# 1. Prepare discovery target lookup once
# ------------------------------------------------------------

target_lookup = {}

for source_name in ["NON_EXTREME", "EXTREME"]:

    subset = (
        discovery_target.loc[
            discovery_target["source_dataset"] == source_name,
            ["source_row", "event"]
        ]
        .sort_values("source_row")
    )

    target_lookup[source_name] = {
        "rows": subset["source_row"].to_numpy(
            dtype=np.int64
        ),
        "events": subset["event"].to_numpy(
            dtype=np.uint8
        )
    }


# ------------------------------------------------------------
# 2. Define screening function
# ------------------------------------------------------------

def screen_feature_batch(batch_index, overwrite=False):

    start_time = time.time()

    selected_features = feature_batches[batch_index]

    batch_number = batch_index + 1

    output_path = os.path.join(
        SCREENING_DIR,
        f"screening_batch_{batch_number:02d}.csv"
    )

    # --------------------------------------------------------
    # Skip completed batches only after basic verification
    # --------------------------------------------------------

    if os.path.isfile(output_path) and not overwrite:

        existing = pd.read_csv(output_path)

        required = {
            "Feature",
            "Condition",
            "Coverage",
            "Event_Probability",
            "Lift"
        }

        if (
            required.issubset(existing.columns)
            and set(existing["Feature"]) == set(selected_features)
            and existing["Feature"].nunique() == len(selected_features)
            and existing["Condition"].notna().all()
            and all(
                set(group["Condition"]) in (
                    {"NO_VALID_READINGS"},
                    {"HIGH_90", "LOW_10", "POSITIVE", "NEGATIVE"}
                )
                for _, group in existing.groupby("Feature")
            )
        ):
            print(f"Batch {batch_number}: Already completed — SKIPPED")
            return existing

        print("Incomplete saved result detected. Recomputing batch.")

    print(f"\n{'=' * 60}")
    print(f"SCREENING BATCH {batch_number}/{len(feature_batches)}")
    print("Features:", len(selected_features))
    print("=" * 60)

    # --------------------------------------------------------
    # 3. Prepare feature storage
    # --------------------------------------------------------

    feature_storage = {
        feature: {
            "values": [],
            "events": []
        }
        for feature in selected_features
    }

    required_columns = []

    for feature in selected_features:

        required_columns.extend([
            feature,
            validity_mapping[feature]
        ])

    # --------------------------------------------------------
    # 4. Process both source datasets
    # --------------------------------------------------------

    for source_name, file_path in [
        ("NON_EXTREME", normal_file),
        ("EXTREME", extreme_file)
    ]:

        pf = pq.ParquetFile(file_path)

        target_rows = target_lookup[source_name]["rows"]
        target_events = target_lookup[source_name]["events"]

        row_offset = 0

        for batch in pf.iter_batches(
            batch_size=32768,
            columns=required_columns,
            use_threads=True
        ):

            batch_end = row_offset + batch.num_rows

            left = np.searchsorted(
                target_rows,
                row_offset
            )

            right = np.searchsorted(
                target_rows,
                batch_end
            )

            if right == left:

                row_offset = batch_end
                continue

            selected_rows = (
                target_rows[left:right] - row_offset
            )

            selected_events = target_events[left:right]

            # ------------------------------------------------
            # Process each feature
            # ------------------------------------------------

            for feature_index, feature in enumerate(
                selected_features
            ):

                value_column = batch.column(
                    feature_index * 2
                )

                valid_column = batch.column(
                    feature_index * 2 + 1
                )

                # Replace missing validity flags with False
                valid_flags = pc.fill_null(
                    valid_column,
                    False
                ).to_numpy(zero_copy_only=False)

                selected_valid = valid_flags[selected_rows]

                if not selected_valid.any():
                    continue

                valid_positions = selected_rows[
                    selected_valid
                ]

                # Retrieve only valid readings
                values_arrow = pc.take(
                    value_column,
                    pa.array(valid_positions)
                )

                values_arrow = pc.cast(
                    values_arrow,
                    pa.float64(),
                    safe=False
                )

                values = values_arrow.to_numpy(
                    zero_copy_only=False
                )

                events = selected_events[selected_valid]

                # Remove null, NaN and infinite readings
                finite_mask = np.isfinite(values)

                if not finite_mask.any():
                    continue

                feature_storage[feature]["values"].append(
                    values[finite_mask]
                )

                feature_storage[feature]["events"].append(
                    events[finite_mask]
                )

            row_offset = batch_end

        assert row_offset == pf.metadata.num_rows

        del pf
        gc.collect()

    # --------------------------------------------------------
    # 5. Calculate statistical screening results
    # --------------------------------------------------------

    screening_results = []

    total_discovery = len(discovery_target)

    for feature in selected_features:

        value_parts = feature_storage[feature]["values"]

        event_parts = feature_storage[feature]["events"]

        if not value_parts:

            screening_results.append({
                "Feature": feature,
                "Condition": "NO_VALID_READINGS",
                "Valid_Observations": 0,
                "Valid_Events": 0,
                "Coverage": 0.0,
                "Valid_Event_Rate": np.nan,
                "Threshold": np.nan,
                "Condition_Observations": 0,
                "Condition_Events": 0,
                "Event_Probability": np.nan,
                "Lift": np.nan
            })

            continue

        values = np.concatenate(value_parts)
        events = np.concatenate(event_parts)

        valid_n = len(values)

        valid_events = int(events.sum())

        valid_event_rate = valid_events / valid_n

        coverage = valid_n / total_discovery

        q10, q90 = np.quantile(
            values,
            [0.10, 0.90]
        )

        conditions = {
            "HIGH_90": (values >= q90, q90),
            "LOW_10": (values <= q10, q10),
            "POSITIVE": (values > 0, 0.0),
            "NEGATIVE": (values < 0, 0.0)
        }

        for condition_name, (mask, threshold) in conditions.items():

            condition_n = int(mask.sum())

            condition_events = int(
                events[mask].sum()
            )

            if condition_n > 0:

                probability = (
                    condition_events / condition_n
                )

                lift = (
                    probability / valid_event_rate
                    if valid_event_rate > 0
                    else np.nan
                )

            else:

                probability = np.nan
                lift = np.nan

            screening_results.append({
                "Feature": feature,
                "Condition": condition_name,
                "Valid_Observations": valid_n,
                "Valid_Events": valid_events,
                "Coverage": coverage,
                "Valid_Event_Rate": valid_event_rate,
                "Threshold": threshold,
                "Condition_Observations": condition_n,
                "Condition_Events": condition_events,
                "Event_Probability": probability,
                "Lift": lift
            })

        # Free feature arrays
        feature_storage[feature]["values"].clear()
        feature_storage[feature]["events"].clear()

        del values, events

    # --------------------------------------------------------
    # 6. Create results DataFrame
    # --------------------------------------------------------

    results_df = pd.DataFrame(screening_results)

    # Add feature metadata
    results_df = results_df.merge(
        feature_catalog_v2[
            [
                "Feature",
                "Family",
                "Parent_Sensor",
                "Transformation",
                "Lookback_1",
                "Lookback_2"
            ]
        ],
        on="Feature",
        how="left",
        validate="many_to_one"
    )

    # Verify all features were processed
    assert set(results_df["Feature"]) == set(selected_features)

    # --------------------------------------------------------
    # 7. Save results atomically
    # --------------------------------------------------------

    # Create file locally before copying it to Drive.
    # This avoids treating a partially written CSV
    # as a completed batch.

    import shutil

    local_path = (
        f"/content/screening_batch_{batch_number:02d}.csv"
    )

    temp_drive_path = output_path + ".tmp"

    results_df.to_csv(local_path, index=False)

    shutil.copy2(local_path, temp_drive_path)

    os.replace(temp_drive_path, output_path)

    # --------------------------------------------------------
    # 8. Display batch summary
    # --------------------------------------------------------

    elapsed = time.time() - start_time

    valid_features = results_df.loc[
        results_df["Condition"] != "NO_VALID_READINGS",
        "Feature"
    ].nunique()

    print("\nBATCH COMPLETED")
    print("-" * 60)

    print("Batch:", batch_number)
    print("Features:", len(selected_features))
    print("Features with valid readings:", valid_features)
    print("Result rows:", len(results_df))
    print(f"Execution time: {elapsed:.2f} seconds")

    print("\nSaved:", output_path)

    del feature_storage
    gc.collect()

    return results_df


print("Reusable screening function is ready.")
print("Total batches available:", len(feature_batches))

Test the Function on Batch 2

In [ ]:
# ============================================================
# STEP 9E.1: TEST REUSABLE FUNCTION ON BATCH 2
# ============================================================

batch_02_results = screen_feature_batch(
    batch_index=1
)

print("\nTOP 10 CONDITIONS — BATCH 2")

display(
    batch_02_results.loc[
        batch_02_results["Condition_Observations"] >= 1000
    ]
    .sort_values(
        "Lift",
        ascending=False
    )
    [
        [
            "Feature",
            "Parent_Sensor",
            "Condition",
            "Coverage",
            "Condition_Observations",
            "Condition_Events",
            "Event_Probability",
            "Lift"
        ]
    ]
    .head(10)
)

Run the Remaining 51 Batches Automatically

Automated Screening

In [ ]:
# ============================================================
# STEP 9F: AUTOMATED SCREENING — REMAINING BATCHES
# ============================================================

import os
import gc
import time
import pandas as pd

overall_start = time.time()

total_batches = len(feature_batches)

completed_batches = []
failed_batches = []

print("AUTOMATED F4Z SCREENING")
print("=" * 65)

print("Total features:", len(f4z_columns))
print("Total batches:", total_batches)
print("Previously completed batches: 2")
print("Starting from Batch 3")

print("=" * 65)

# ------------------------------------------------------------
# Execute remaining batches
# ------------------------------------------------------------

for batch_index in range(2, total_batches):

    batch_number = batch_index + 1

    print(
        f"\nProgress: Batch {batch_number}/{total_batches}"
    )

    try:

        batch_result = screen_feature_batch(
            batch_index=batch_index,
            overwrite=False
        )

        # Verify current batch
        expected_features = set(
            feature_batches[batch_index]
        )

        actual_features = set(
            batch_result["Feature"]
        )

        assert expected_features == actual_features, (
            f"Feature mismatch in Batch {batch_number}"
        )

        completed_batches.append(batch_number)

        # Release result dataframe immediately
        del batch_result
        gc.collect()

    except Exception as e:

        failed_batches.append(batch_number)

        print(
            f"\nERROR in Batch {batch_number}: {e}"
        )

        print(
            "Stopping to prevent unnoticed missing results."
        )

        break

    # --------------------------------------------------------
    # Progress checkpoint every 5 batches
    # --------------------------------------------------------

    if batch_number % 5 == 0:

        elapsed = time.time() - overall_start

        print("\nPROGRESS CHECKPOINT")
        print("-" * 50)

        print(
            "Latest completed batch:",
            batch_number
        )

        print(
            "Batches completed in this run:",
            len(completed_batches)
        )

        print(
            "Elapsed time:",
            f"{elapsed / 60:.2f} minutes"
        )

# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("AUTOMATED SCREENING SUMMARY")
print("=" * 65)

saved_batch_files = []

for batch_number in range(1, total_batches + 1):

    batch_path = os.path.join(
        SCREENING_DIR,
        f"screening_batch_{batch_number:02d}.csv"
    )

    if os.path.isfile(batch_path):

        saved_batch_files.append(batch_path)

print("Expected batch files:", total_batches)

print("Saved batch files:", len(saved_batch_files))

print("Batches completed in this run:", len(completed_batches))

print("Failed batches:", failed_batches)

print(
    "Total elapsed time:",
    f"{(time.time() - overall_start) / 60:.2f} minutes"
)

if len(saved_batch_files) == total_batches and not failed_batches:

    print("\nALL BATCH FILES FOUND")

else:

    print("\nSCREENING INCOMPLETE — REVIEW MISSING BATCHES")

Combine and Audit Screening Results

In [ ]:
# ============================================================
# STEP 10A: COMBINE AND AUDIT ALL SCREENING RESULTS
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Load all 53 batch result files
# ------------------------------------------------------------

all_batch_results = []

print("LOADING SCREENING RESULTS")
print("=" * 65)

for batch_number in range(1, 54):

    filename = f"screening_batch_{batch_number:02d}.csv"

    path = os.path.join(
        SCREENING_DIR,
        filename
    )

    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"Missing screening batch: {filename}"
        )

    df = pd.read_csv(path)

    df["Batch_Number"] = batch_number

    all_batch_results.append(df)

# Combine
full_screening_results = pd.concat(
    all_batch_results,
    ignore_index=True
)

del all_batch_results

print("Batch files loaded:", 53)

print(
    "Total screening result rows:",
    len(full_screening_results)
)

# ------------------------------------------------------------
# 2. Verify feature completeness
# ------------------------------------------------------------

expected_features = set(f4z_columns)

observed_features = set(
    full_screening_results["Feature"]
)

missing_features = expected_features - observed_features

unexpected_features = observed_features - expected_features

print("\nFEATURE COMPLETENESS")
print("=" * 65)

print("Expected features:", len(expected_features))
print("Observed features:", len(observed_features))
print("Missing features:", len(missing_features))
print("Unexpected features:", len(unexpected_features))

assert len(missing_features) == 0
assert len(unexpected_features) == 0

# ------------------------------------------------------------
# 3. Verify duplicate feature-condition records
# ------------------------------------------------------------

duplicates = full_screening_results.duplicated(
    subset=["Feature", "Condition"]
).sum()

print("\nDUPLICATE CHECK")

print("Duplicate feature-condition records:", duplicates)

assert duplicates == 0

# ------------------------------------------------------------
# 4. Verify condition structure
# ------------------------------------------------------------

expected_conditions = {
    "HIGH_90",
    "LOW_10",
    "POSITIVE",
    "NEGATIVE"
}

invalid_structures = []

for feature, group in full_screening_results.groupby("Feature"):

    conditions = set(group["Condition"])

    if conditions not in (
        expected_conditions,
        {"NO_VALID_READINGS"}
    ):

        invalid_structures.append(feature)

print("\nCONDITION STRUCTURE")

print("Features with invalid condition structure:",
      len(invalid_structures))

assert len(invalid_structures) == 0

# ------------------------------------------------------------
# 5. Count usable and unusable features
# ------------------------------------------------------------

no_valid_features = (
    full_screening_results.loc[
        full_screening_results["Condition"] == "NO_VALID_READINGS",
        "Feature"
    ].nunique()
)

usable_features = (
    len(observed_features) - no_valid_features
)

print("\nFEATURE USABILITY")

print("Features with usable readings:", usable_features)
print("Features without usable readings:", no_valid_features)

# ------------------------------------------------------------
# 6. Validate numerical metrics
# ------------------------------------------------------------

valid_results = full_screening_results.loc[
    full_screening_results["Condition"] != "NO_VALID_READINGS"
].copy()

# Coverage must be between 0 and 1
assert valid_results["Coverage"].between(0, 1).all()

# Condition counts cannot exceed valid observations
assert (
    valid_results["Condition_Observations"]
    <= valid_results["Valid_Observations"]
).all()

# Event counts cannot exceed condition observations
assert (
    valid_results["Condition_Events"]
    <= valid_results["Condition_Observations"]
).all()

# Positive-event counts cannot exceed valid observations
if "Valid_Events" in valid_results.columns:

    valid_event_counts = pd.to_numeric(
        valid_results["Valid_Events"],
        errors="coerce"
    )

    available_counts = valid_event_counts.notna()

    assert (
        valid_event_counts[available_counts]
        <= valid_results.loc[
            available_counts,
            "Valid_Observations"
        ]
    ).all()

# Event probability must equal events / observations
nonempty = (
    valid_results["Condition_Observations"] > 0
)

calculated_probabilities = (
    valid_results.loc[
        nonempty,
        "Condition_Events"
    ]
    /
    valid_results.loc[
        nonempty,
        "Condition_Observations"
    ]
)

assert np.allclose(
    calculated_probabilities,
    valid_results.loc[
        nonempty,
        "Event_Probability"
    ],
    rtol=1e-9,
    atol=1e-9
)

print("\nNUMERICAL CHECKS: PASSED")

# ------------------------------------------------------------
# 7. Summary by family
# ------------------------------------------------------------

family_summary = (
    full_screening_results[
        ["Feature", "Family"]
    ]
    .drop_duplicates()
    .groupby("Family")
    .agg(
        Total_Features=("Feature", "count")
    )
    .reset_index()
)

print("\nFEATURES BY FAMILY")

display(family_summary)

# ------------------------------------------------------------
# 8. Save combined screening results
# ------------------------------------------------------------

combined_path = os.path.join(
    SCREENING_DIR,
    "all_f4z_screening_results.csv"
)

full_screening_results.to_csv(
    combined_path,
    index=False
)

print("\nCOMBINED RESULTS SAVED")

print(combined_path)

print("\nSCREENING AUDIT COMPLETED")

Preliminary Feature Ranking

In [ ]:
# ============================================================
# STEP 10B: PRELIMINARY F4Z FEATURE RANKING
# ============================================================

import numpy as np
import pandas as pd
import os

# ------------------------------------------------------------
# 1. Load combined screening results
# ------------------------------------------------------------

screening_path = os.path.join(
    SCREENING_DIR,
    "all_f4z_screening_results.csv"
)

ranking_df = pd.read_csv(screening_path)

# ------------------------------------------------------------
# 2. Exclude conditions without usable observations
# ------------------------------------------------------------

ranking_df = ranking_df.loc[
    (ranking_df["Condition"] != "NO_VALID_READINGS")
    &
    (ranking_df["Condition_Observations"] > 0)
    &
    ranking_df["Event_Probability"].notna()
].copy()

print("PRELIMINARY FEATURE RANKING")
print("=" * 65)

print("Usable feature-condition records:", len(ranking_df))

print("Unique features:", ranking_df["Feature"].nunique())

# ------------------------------------------------------------
# 3. Calculate overall discovery lift
# ------------------------------------------------------------

DISCOVERY_BASE_RATE = (
    discovery_target["event"].mean()
)

ranking_df["Overall_Lift"] = (
    ranking_df["Event_Probability"]
    / DISCOVERY_BASE_RATE
)

# ------------------------------------------------------------
# 4. Calculate condition prevalence
# ------------------------------------------------------------

ranking_df["Condition_Prevalence"] = (
    ranking_df["Condition_Observations"]
    / len(discovery_target)
)

ranking_df["Condition_Share_Of_Valid"] = (
    ranking_df["Condition_Observations"]
    / ranking_df["Valid_Observations"]
)

# ------------------------------------------------------------
# 5. Define preliminary evidence tiers
# ------------------------------------------------------------

# These are screening filters, not statistical
# significance tests or final selection criteria.

MIN_CONDITION_OBSERVATIONS = 1000

MIN_COVERAGE = 0.10

ranking_df["Preliminary_Eligible"] = (
    (ranking_df["Condition_Observations"]
     >= MIN_CONDITION_OBSERVATIONS)
    &
    (ranking_df["Coverage"] >= MIN_COVERAGE)
)

# ------------------------------------------------------------
# 6. Rank eligible conditions
# ------------------------------------------------------------

eligible_results = ranking_df.loc[
    ranking_df["Preliminary_Eligible"]
].copy()

eligible_results = eligible_results.sort_values(
    [
        "Lift",
        "Condition_Events",
        "Coverage"
    ],
    ascending=[False, False, False]
)

# ------------------------------------------------------------
# 7. Display top 25 conditions
# ------------------------------------------------------------

print("\nTOP 25 FEATURE CONDITIONS BY VALID-BASELINE LIFT")

display(
    eligible_results[
        [
            "Feature",
            "Family",
            "Parent_Sensor",
            "Condition",
            "Coverage",
            "Condition_Observations",
            "Condition_Events",
            "Event_Probability",
            "Lift",
            "Overall_Lift"
        ]
    ].head(25).round(4)
)

# ------------------------------------------------------------
# 8. Identify best condition per feature
# ------------------------------------------------------------

best_per_feature = (
    eligible_results
    .drop_duplicates(
        subset=["Feature"],
        keep="first"
    )
    .copy()
)

print("\nTOP 20 DISTINCT F4Z COLUMNS")

display(
    best_per_feature[
        [
            "Feature",
            "Family",
            "Parent_Sensor",
            "Condition",
            "Coverage",
            "Condition_Observations",
            "Event_Probability",
            "Lift",
            "Overall_Lift"
        ]
    ].head(20).round(4)
)

# ------------------------------------------------------------
# 9. Identify strongest condition per parent sensor
# ------------------------------------------------------------

# Group by family AND parent sensor to avoid
# combining similarly named sensors across families.

best_per_parent = (
    eligible_results
    .drop_duplicates(
        subset=["Family", "Parent_Sensor"],
        keep="first"
    )
    .copy()
)

print("\nTOP 15 PARENT SENSORS")

display(
    best_per_parent[
        [
            "Family",
            "Parent_Sensor",
            "Feature",
            "Condition",
            "Coverage",
            "Event_Probability",
            "Lift",
            "Overall_Lift"
        ]
    ].head(15).round(4)
)

# ------------------------------------------------------------
# 10. Save preliminary rankings
# ------------------------------------------------------------

ranking_df.to_csv(
    os.path.join(
        SCREENING_DIR,
        "all_conditions_ranked.csv"
    ),
    index=False
)

best_per_feature.to_csv(
    os.path.join(
        SCREENING_DIR,
        "best_condition_per_feature.csv"
    ),
    index=False
)

best_per_parent.to_csv(
    os.path.join(
        SCREENING_DIR,
        "best_condition_per_parent.csv"
    ),
    index=False
)

print("\nPRELIMINARY RANKINGS SAVED")

Examine Candidate Coverage and Baseline Differences

Candidate Evidence Summary

In [ ]:
# ============================================================
# STEP 11A: CANDIDATE EVIDENCE SUMMARY
# ============================================================

import os
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Load saved parent-sensor rankings
# ------------------------------------------------------------

parent_path = os.path.join(
    SCREENING_DIR,
    "best_condition_per_parent.csv"
)

candidate_df = pd.read_csv(parent_path)

# ------------------------------------------------------------
# Select the strongest 15 parent-sensor candidates
# ------------------------------------------------------------

candidates = candidate_df.head(15).copy()

# ------------------------------------------------------------
# Calculate baseline and prevalence statistics
# ------------------------------------------------------------

candidates["Valid_Baseline_Percent"] = (
    candidates["Valid_Event_Rate"] * 100
)

candidates["Event_Probability_Percent"] = (
    candidates["Event_Probability"] * 100
)

candidates["Coverage_Percent"] = (
    candidates["Coverage"] * 100
)

candidates["Condition_Prevalence_Percent"] = (
    candidates["Condition_Prevalence"] * 100
)

# Difference in percentage points
candidates["Absolute_Event_Increase_pp"] = (
    candidates["Event_Probability_Percent"]
    - candidates["Valid_Baseline_Percent"]
)

# ------------------------------------------------------------
# Display full feature names without truncation
# ------------------------------------------------------------

pd.set_option("display.max_colwidth", None)

print("TOP 15 PARENT-SENSOR CANDIDATES")
print("=" * 70)

display(
    candidates[
        [
            "Family",
            "Parent_Sensor",
            "Feature",
            "Condition",
            "Coverage_Percent",
            "Condition_Observations",
            "Condition_Events",
            "Valid_Baseline_Percent",
            "Event_Probability_Percent",
            "Absolute_Event_Increase_pp",
            "Lift",
            "Overall_Lift"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# Save candidate summary
# ------------------------------------------------------------

candidate_output_path = os.path.join(
    SCREENING_DIR,
    "top_15_parent_candidate_summary.csv"
)

candidates.to_csv(
    candidate_output_path,
    index=False
)

print("\nCandidate summary saved:")
print(candidate_output_path)

Quarterly Stability Analysis

In [ ]:
# ============================================================
# STEP 11B: QUARTERLY STABILITY — TOP 5 PARENT SENSORS
# ============================================================

import os
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

# ------------------------------------------------------------
# 1. Select top 5 distinct parent-sensor candidates
# ------------------------------------------------------------

selected_candidates = candidates.head(5).copy()

required_candidate_fields = [
    "Feature", "Condition", "Threshold"
]

assert all(
    c in selected_candidates.columns
    for c in required_candidate_fields
)

assert selected_candidates["Feature"].is_unique

print("SELECTED CANDIDATES")
print("=" * 65)

display(
    selected_candidates[
        ["Family", "Parent_Sensor", "Feature",
         "Condition", "Threshold", "Lift"]
    ]
)

# ------------------------------------------------------------
# 2. Create quarterly labels for discovery observations
# ------------------------------------------------------------

target_quarters = discovery_target[
    ["source_dataset", "source_row", "decision_time", "event"]
].copy()

target_quarters["Quarter"] = (
    target_quarters["decision_time"].dt.year.astype(str)
    + "-Q"
    + target_quarters["decision_time"].dt.quarter.astype(str)
)

all_quarters = sorted(
    target_quarters["Quarter"].unique()
)

# ------------------------------------------------------------
# 3. Prepare source-row lookups
# ------------------------------------------------------------

quarter_lookup = {}

for source_name in ["NON_EXTREME", "EXTREME"]:

    subset = (
        target_quarters.loc[
            target_quarters["source_dataset"] == source_name
        ]
        .sort_values("source_row")
    )

    quarter_lookup[source_name] = {
        "rows": subset["source_row"].to_numpy(dtype=np.int64),
        "events": subset["event"].to_numpy(dtype=np.uint8),
        "quarters": subset["Quarter"].to_numpy()
    }

# Discovery observation count for each quarter
quarter_totals = (
    target_quarters.groupby("Quarter")
    .size()
    .to_dict()
)

# ------------------------------------------------------------
# 4. Initialize quarterly counters
# ------------------------------------------------------------

stats = {}

for feature in selected_candidates["Feature"]:

    for quarter in all_quarters:

        stats[(feature, quarter)] = {
            "Valid_N": 0,
            "Valid_Events": 0,
            "Condition_N": 0,
            "Condition_Events": 0
        }

candidate_info = selected_candidates.set_index("Feature")

# ------------------------------------------------------------
# 5. Process source datasets in PyArrow batches
# ------------------------------------------------------------

feature_names = selected_candidates["Feature"].tolist()

columns_to_read = []

for feature in feature_names:
    columns_to_read.extend([
        feature,
        validity_mapping[feature]
    ])

for source_name, file_path in [
    ("NON_EXTREME", normal_file),
    ("EXTREME", extreme_file)
]:

    print(f"\nProcessing {source_name}...")

    pf = pq.ParquetFile(file_path)

    lookup = quarter_lookup[source_name]

    source_rows = lookup["rows"]
    source_events = lookup["events"]
    source_quarters = lookup["quarters"]

    row_offset = 0

    for batch in pf.iter_batches(
        batch_size=32768,
        columns=columns_to_read,
        use_threads=True
    ):

        batch_end = row_offset + batch.num_rows

        left = np.searchsorted(source_rows, row_offset)
        right = np.searchsorted(source_rows, batch_end)

        if right == left:
            row_offset = batch_end
            continue

        positions = source_rows[left:right] - row_offset
        events = source_events[left:right]
        quarters = source_quarters[left:right]

        for i, feature in enumerate(feature_names):

            value_col = batch.column(i * 2)
            valid_col = batch.column(i * 2 + 1)

            flags = pc.fill_null(
                valid_col, False
            ).to_numpy(zero_copy_only=False)

            selected_valid = flags[positions]

            if not selected_valid.any():
                continue

            selected_positions = positions[selected_valid]

            values_arrow = pc.take(
                value_col,
                pa.array(selected_positions)
            )

            values_arrow = pc.cast(
                values_arrow,
                pa.float64(),
                safe=False
            )

            values = values_arrow.to_numpy(
                zero_copy_only=False
            )

            selected_events = events[selected_valid]
            selected_quarters = quarters[selected_valid]

            finite = np.isfinite(values)

            values = values[finite]
            selected_events = selected_events[finite]
            selected_quarters = selected_quarters[finite]

            condition = candidate_info.loc[feature, "Condition"]
            threshold = float(
                candidate_info.loc[feature, "Threshold"]
            )

            if condition == "HIGH_90":
                mask = values >= threshold

            elif condition == "LOW_10":
                mask = values <= threshold

            elif condition == "POSITIVE":
                mask = values > 0

            elif condition == "NEGATIVE":
                mask = values < 0

            else:
                raise ValueError(
                    f"Unexpected condition: {condition}"
                )

            for quarter in np.unique(selected_quarters):

                qmask = selected_quarters == quarter
                cmask = qmask & mask

                s = stats[(feature, quarter)]

                s["Valid_N"] += int(qmask.sum())
                s["Valid_Events"] += int(
                    selected_events[qmask].sum()
                )

                s["Condition_N"] += int(cmask.sum())
                s["Condition_Events"] += int(
                    selected_events[cmask].sum()
                )

        row_offset = batch_end

    assert row_offset == pf.metadata.num_rows

    print("Completed.")

# ------------------------------------------------------------
# 6. Calculate quarterly statistics
# ------------------------------------------------------------

quarterly_rows = []

for (feature, quarter), s in stats.items():

    valid_n = s["Valid_N"]
    valid_events = s["Valid_Events"]

    condition_n = s["Condition_N"]
    condition_events = s["Condition_Events"]

    valid_rate = (
        valid_events / valid_n
        if valid_n > 0 else np.nan
    )

    condition_rate = (
        condition_events / condition_n
        if condition_n > 0 else np.nan
    )

    lift = (
        condition_rate / valid_rate
        if valid_rate > 0 and condition_n > 0
        else np.nan
    )

    quarterly_rows.append({
        "Feature": feature,
        "Quarter": quarter,
        "Parent_Sensor": candidate_info.loc[
            feature, "Parent_Sensor"
        ],
        "Condition": candidate_info.loc[
            feature, "Condition"
        ],
        "Valid_N": valid_n,
        "Valid_Events": valid_events,
        "Coverage": valid_n / quarter_totals[quarter],
        "Condition_N": condition_n,
        "Condition_Events": condition_events,
        "Valid_Event_Rate": valid_rate,
        "Event_Probability": condition_rate,
        "Lift": lift
    })

quarterly_results = pd.DataFrame(quarterly_rows)

# ------------------------------------------------------------
# 7. Save results
# ------------------------------------------------------------

output_path = os.path.join(
    SCREENING_DIR,
    "top5_quarterly_stability.csv"
)

quarterly_results.to_csv(
    output_path,
    index=False
)

# ------------------------------------------------------------
# 8. Display results
# ------------------------------------------------------------

print("\nQUARTERLY STABILITY RESULTS")
print("=" * 65)

display(
    quarterly_results[
        [
            "Parent_Sensor",
            "Quarter",
            "Coverage",
            "Valid_N",
            "Condition_N",
            "Condition_Events",
            "Valid_Event_Rate",
            "Event_Probability",
            "Lift"
        ]
    ].round(4)
)

print("\nQUARTERLY LIFT COMPARISON")

lift_pivot = quarterly_results.pivot(
    index="Quarter",
    columns="Parent_Sensor",
    values="Lift"
)

display(lift_pivot.round(3))

print("\nResults saved:")
print(output_path)

Prepare Discovery Episode Mapping

In [ ]:
# ============================================================
# STEP 11C: PREPARE DISCOVERY EPISODE MAPPING
# ============================================================

import os
import numpy as np
import pandas as pd

# Load saved discovery target
target_path = os.path.join(
    SCREENING_DIR,
    "discovery_target.parquet"
)

episode_target = pd.read_parquet(target_path)

episode_target = (
    episode_target
    .sort_values("decision_time")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 1. Identify positive-event observations
# ------------------------------------------------------------

episode_target["event"] = (
    episode_target["signed_return"] > 500
).astype(np.uint8)

# ------------------------------------------------------------
# 2. Calculate timestamp gaps
# ------------------------------------------------------------

time_diff = (
    episode_target["decision_time"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

# ------------------------------------------------------------
# 3. Identify starts of positive-event runs
# ------------------------------------------------------------

previous_event = (
    episode_target["event"]
    .shift(1)
    .fillna(0)
)

new_positive_run = (
    (episode_target["event"] == 1)
    &
    (
        (previous_event != 1)
        |
        (time_diff != 1)
    )
)

# ------------------------------------------------------------
# 4. Assign positive-run identifiers
# ------------------------------------------------------------

run_ids = new_positive_run.cumsum()

episode_target["Positive_Run_ID"] = np.where(
    episode_target["event"] == 1,
    run_ids,
    0
)

# ------------------------------------------------------------
# 5. Assign fixed 48-hour UTC blocks
# ------------------------------------------------------------

origin = episode_target["decision_time"].min()

elapsed_hours = (
    episode_target["decision_time"] - origin
).dt.total_seconds() / 3600

episode_target["Block_48h"] = (
    elapsed_hours // 48
).astype(np.int32)

# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------

positive_rows = episode_target.loc[
    episode_target["event"] == 1
]

print("DISCOVERY EPISODE MAPPING")
print("=" * 65)

print(
    "Discovery observations:",
    f"{len(episode_target):,}"
)

print(
    "Positive-event observations:",
    f"{len(positive_rows):,}"
)

print(
    "Positive-event runs:",
    positive_rows["Positive_Run_ID"].nunique()
)

print(
    "48-hour blocks containing positive events:",
    positive_rows["Block_48h"].nunique()
)

# ------------------------------------------------------------
# 7. Save mapping
# ------------------------------------------------------------

episode_mapping_path = os.path.join(
    SCREENING_DIR,
    "discovery_episode_mapping.parquet"
)

episode_target[
    [
        "decision_time",
        "source_dataset",
        "source_row",
        "event",
        "Positive_Run_ID",
        "Block_48h"
    ]
].to_parquet(
    episode_mapping_path,
    index=False
)

print("\nEpisode mapping saved:")
print(episode_mapping_path)

print("\nSAMPLE POSITIVE-EVENT OBSERVATIONS")

display(
    positive_rows[
        [
            "decision_time",
            "event",
            "Positive_Run_ID",
            "Block_48h"
        ]
    ].head(10)
)

Calculate Episode Support

Episode-Support Analysis

In [ ]:
# ============================================================
# STEP 11D: EPISODE SUPPORT — TOP 5 F4Z CANDIDATES
# ============================================================

import os
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

# ------------------------------------------------------------
# 1. Load saved candidate definitions
# ------------------------------------------------------------

candidate_path = os.path.join(
    SCREENING_DIR,
    "top_15_parent_candidate_summary.csv"
)

episode_candidates = pd.read_csv(candidate_path).head(5).copy()

feature_names = episode_candidates["Feature"].tolist()

assert len(feature_names) == 5
assert len(set(feature_names)) == 5

candidate_lookup = episode_candidates.set_index("Feature")

# ------------------------------------------------------------
# 2. Load existing discovery episode mapping
# ------------------------------------------------------------

mapping_path = os.path.join(
    SCREENING_DIR,
    "discovery_episode_mapping.parquet"
)

episode_mapping = pd.read_parquet(mapping_path)

# Prepare source-row lookups
source_lookup = {}

for source_name in ["NON_EXTREME", "EXTREME"]:

    subset = (
        episode_mapping.loc[
            episode_mapping["source_dataset"] == source_name
        ]
        .sort_values("source_row")
    )

    source_lookup[source_name] = {
        "rows": subset["source_row"].to_numpy(dtype=np.int64),
        "events": subset["event"].to_numpy(dtype=np.uint8),
        "run_ids": subset["Positive_Run_ID"].to_numpy(
            dtype=np.int64
        ),
        "blocks": subset["Block_48h"].to_numpy(
            dtype=np.int32
        )
    }

# ------------------------------------------------------------
# 3. Initialize episode-support counters
# ------------------------------------------------------------

block_count = int(episode_mapping["Block_48h"].max()) + 1

support = {}

for feature in feature_names:

    support[feature] = {
        "Condition_N": 0,
        "Condition_Events": 0,
        "Positive_Run_IDs": set(),
        "Positive_Block_Counts": np.zeros(
            block_count, dtype=np.int64
        )
    }

# ------------------------------------------------------------
# 4. Read only the five features and validity columns
# ------------------------------------------------------------

columns_to_read = []

for feature in feature_names:

    columns_to_read.extend([
        feature,
        validity_mapping[feature]
    ])

for source_name, file_path in [
    ("NON_EXTREME", normal_file),
    ("EXTREME", extreme_file)
]:

    print(f"\nProcessing {source_name}...")

    pf = pq.ParquetFile(file_path)

    lookup = source_lookup[source_name]

    source_rows = lookup["rows"]
    source_events = lookup["events"]
    source_runs = lookup["run_ids"]
    source_blocks = lookup["blocks"]

    row_offset = 0

    for batch in pf.iter_batches(
        batch_size=32768,
        columns=columns_to_read,
        use_threads=True
    ):

        batch_end = row_offset + batch.num_rows

        left = np.searchsorted(source_rows, row_offset)
        right = np.searchsorted(source_rows, batch_end)

        if right == left:
            row_offset = batch_end
            continue

        positions = source_rows[left:right] - row_offset

        events = source_events[left:right]
        runs = source_runs[left:right]
        blocks = source_blocks[left:right]

        # Process each candidate independently
        for i, feature in enumerate(feature_names):

            value_col = batch.column(i * 2)
            valid_col = batch.column(i * 2 + 1)

            valid_flags = pc.fill_null(
                valid_col, False
            ).to_numpy(zero_copy_only=False)

            selected_valid = valid_flags[positions]

            if not selected_valid.any():
                continue

            valid_positions = positions[selected_valid]

            values_arrow = pc.take(
                value_col,
                pa.array(valid_positions)
            )

            values_arrow = pc.cast(
                values_arrow,
                pa.float64(),
                safe=False
            )

            values = values_arrow.to_numpy(
                zero_copy_only=False
            )

            selected_events = events[selected_valid]
            selected_runs = runs[selected_valid]
            selected_blocks = blocks[selected_valid]

            finite = np.isfinite(values)

            values = values[finite]
            selected_events = selected_events[finite]
            selected_runs = selected_runs[finite]
            selected_blocks = selected_blocks[finite]

            condition = candidate_lookup.loc[
                feature, "Condition"
            ]

            threshold = float(
                candidate_lookup.loc[feature, "Threshold"]
            )

            # Apply original discovery-period condition
            if condition == "HIGH_90":
                condition_mask = values >= threshold

            elif condition == "LOW_10":
                condition_mask = values <= threshold

            elif condition == "POSITIVE":
                condition_mask = values > 0

            elif condition == "NEGATIVE":
                condition_mask = values < 0

            else:
                raise ValueError(
                    f"Unknown condition: {condition}"
                )

            s = support[feature]

            s["Condition_N"] += int(condition_mask.sum())

            positive_mask = (
                condition_mask & (selected_events == 1)
            )

            s["Condition_Events"] += int(
                positive_mask.sum()
            )

            # Positive-event runs supporting the condition
            positive_runs = selected_runs[positive_mask]

            s["Positive_Run_IDs"].update(
                np.unique(positive_runs[positive_runs > 0]).tolist()
            )

            # Count positive observations by 48-hour block
            positive_blocks = selected_blocks[positive_mask]

            if len(positive_blocks):

                s["Positive_Block_Counts"] += np.bincount(
                    positive_blocks,
                    minlength=block_count
                )

        row_offset = batch_end

    assert row_offset == pf.metadata.num_rows

    print("Completed.")

# ------------------------------------------------------------
# 5. Calculate episode-support statistics
# ------------------------------------------------------------

summary_rows = []

for feature in feature_names:

    s = support[feature]

    condition_n = s["Condition_N"]
    condition_events = s["Condition_Events"]

    block_counts = s["Positive_Block_Counts"]

    supported_blocks = int(
        np.count_nonzero(block_counts)
    )

    largest_block = int(
        block_counts.max()
    )

    largest_block_share = (
        largest_block / condition_events
        if condition_events > 0
        else np.nan
    )

    summary_rows.append({
        "Feature": feature,
        "Parent_Sensor": candidate_lookup.loc[
            feature, "Parent_Sensor"
        ],
        "Condition": candidate_lookup.loc[
            feature, "Condition"
        ],
        "Condition_Observations": condition_n,
        "Condition_Events": condition_events,
        "Supported_Positive_Runs": len(
            s["Positive_Run_IDs"]
        ),
        "Supported_48h_Blocks": supported_blocks,
        "Largest_Block_Event_Count": largest_block,
        "Largest_Block_Share": largest_block_share
    })

episode_support_results = pd.DataFrame(summary_rows)

# ------------------------------------------------------------
# 6. Verify counts against original screening
# ------------------------------------------------------------

for _, row in episode_support_results.iterrows():

    original = candidate_lookup.loc[row["Feature"]]

    assert (
        row["Condition_Observations"]
        == int(original["Condition_Observations"])
    ), f"Condition count mismatch: {row['Feature']}"

    assert (
        row["Condition_Events"]
        == int(original["Condition_Events"])
    ), f"Event count mismatch: {row['Feature']}"

print("\nSCREENING COUNT CONSISTENCY: PASSED")

# ------------------------------------------------------------
# 7. Display results
# ------------------------------------------------------------

print("\nEPISODE SUPPORT — TOP 5 CANDIDATES")
print("=" * 70)

display(
    episode_support_results.round(4)
)

# ------------------------------------------------------------
# 8. Save results
# ------------------------------------------------------------

output_path = os.path.join(
    SCREENING_DIR,
    "top5_episode_support.csv"
)

episode_support_results.to_csv(
    output_path,
    index=False
)

print("\nEpisode-support results saved:")
print(output_path)

Select Five Alternative Parent Sensors

Prepare Alternative Candidates

In [ ]:
# ============================================================
# STEP 11E: PREPARE FIVE ALTERNATIVE CANDIDATES
# ============================================================

import os
import pandas as pd

# Load previously saved parent-sensor rankings
parent_path = os.path.join(
    SCREENING_DIR,
    "best_condition_per_parent.csv"
)

parent_rankings = pd.read_csv(parent_path)

# Select five additional parent sensors
alternative_sensors = [
    ("funding", "funding_range"),
    ("flow", "flow_activity"),
    ("oi", "oi_extreme_recency"),
    ("price", "price_displacement"),
    ("basis", "basis_range")
]

alternative_rows = []

for family, sensor in alternative_sensors:

    match = parent_rankings.loc[
        (parent_rankings["Family"] == family)
        &
        (parent_rankings["Parent_Sensor"] == sensor)
    ]

    assert len(match) == 1, (
        f"Expected one candidate for {family}/{sensor}"
    )

    alternative_rows.append(match.iloc[0])

alternative_candidates = pd.DataFrame(
    alternative_rows
).reset_index(drop=True)

# Display complete feature names
pd.set_option("display.max_colwidth", None)

print("ALTERNATIVE F4Z CANDIDATES")
print("=" * 70)

display(
    alternative_candidates[
        [
            "Family",
            "Parent_Sensor",
            "Feature",
            "Condition",
            "Threshold",
            "Coverage",
            "Condition_Observations",
            "Condition_Events",
            "Event_Probability",
            "Lift",
            "Overall_Lift"
        ]
    ].round(4)
)

# Save the candidate selection
output_path = os.path.join(
    SCREENING_DIR,
    "alternative_5_candidates.csv"
)

alternative_candidates.to_csv(
    output_path,
    index=False
)

print("\nAlternative candidates saved:")
print(output_path)

Evaluate Quarterly Stability

Reusable Quarterly Stability Function

In [ ]:
# ============================================================
# STEP 11F: REUSABLE QUARTERLY STABILITY FUNCTION
# ============================================================

import os
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc


def calculate_quarterly_stability(
    selected_candidates,
    output_filename
):

    selected_candidates = selected_candidates.copy()

    assert selected_candidates["Feature"].is_unique

    feature_names = selected_candidates["Feature"].tolist()

    candidate_info = selected_candidates.set_index("Feature")

    # --------------------------------------------------------
    # 1. Prepare discovery-period quarterly labels
    # --------------------------------------------------------

    target_data = discovery_target[
        [
            "source_dataset",
            "source_row",
            "decision_time",
            "event"
        ]
    ].copy()

    target_data["Quarter"] = (
        target_data["decision_time"].dt.year.astype(str)
        + "-Q"
        + target_data["decision_time"].dt.quarter.astype(str)
    )

    all_quarters = sorted(
        target_data["Quarter"].unique()
    )

    quarter_totals = (
        target_data.groupby("Quarter")
        .size()
        .to_dict()
    )

    source_lookup = {}

    for source_name in ["NON_EXTREME", "EXTREME"]:

        subset = (
            target_data.loc[
                target_data["source_dataset"] == source_name
            ]
            .sort_values("source_row")
        )

        source_lookup[source_name] = {
            "rows": subset["source_row"].to_numpy(
                dtype=np.int64
            ),
            "events": subset["event"].to_numpy(
                dtype=np.uint8
            ),
            "quarters": subset["Quarter"].to_numpy()
        }

    # --------------------------------------------------------
    # 2. Initialize statistics
    # --------------------------------------------------------

    stats = {}

    for feature in feature_names:

        for quarter in all_quarters:

            stats[(feature, quarter)] = {
                "Valid_N": 0,
                "Valid_Events": 0,
                "Condition_N": 0,
                "Condition_Events": 0
            }

    # --------------------------------------------------------
    # 3. Select necessary Parquet columns
    # --------------------------------------------------------

    columns_to_read = []

    for feature in feature_names:

        columns_to_read.extend([
            feature,
            validity_mapping[feature]
        ])

    # --------------------------------------------------------
    # 4. Process source datasets
    # --------------------------------------------------------

    for source_name, file_path in [
        ("NON_EXTREME", normal_file),
        ("EXTREME", extreme_file)
    ]:

        print(f"\nProcessing {source_name}...")

        pf = pq.ParquetFile(file_path)

        lookup = source_lookup[source_name]

        source_rows = lookup["rows"]
        source_events = lookup["events"]
        source_quarters = lookup["quarters"]

        row_offset = 0

        for batch in pf.iter_batches(
            batch_size=32768,
            columns=columns_to_read,
            use_threads=True
        ):

            batch_end = row_offset + batch.num_rows

            left = np.searchsorted(
                source_rows, row_offset
            )

            right = np.searchsorted(
                source_rows, batch_end
            )

            if right == left:
                row_offset = batch_end
                continue

            positions = (
                source_rows[left:right] - row_offset
            )

            events = source_events[left:right]

            quarters = source_quarters[left:right]

            # ------------------------------------------------
            # Evaluate each feature
            # ------------------------------------------------

            for i, feature in enumerate(feature_names):

                value_col = batch.column(i * 2)
                valid_col = batch.column(i * 2 + 1)

                flags = pc.fill_null(
                    valid_col,
                    False
                ).to_numpy(zero_copy_only=False)

                selected_valid = flags[positions]

                if not selected_valid.any():
                    continue

                selected_positions = positions[
                    selected_valid
                ]

                values_arrow = pc.take(
                    value_col,
                    pa.array(selected_positions)
                )

                values_arrow = pc.cast(
                    values_arrow,
                    pa.float64(),
                    safe=False
                )

                values = values_arrow.to_numpy(
                    zero_copy_only=False
                )

                selected_events = events[selected_valid]

                selected_quarters = quarters[selected_valid]

                finite = np.isfinite(values)

                values = values[finite]

                selected_events = selected_events[finite]

                selected_quarters = selected_quarters[finite]

                condition = candidate_info.loc[
                    feature, "Condition"
                ]

                threshold = float(
                    candidate_info.loc[
                        feature, "Threshold"
                    ]
                )

                if condition == "HIGH_90":
                    condition_mask = values >= threshold

                elif condition == "LOW_10":
                    condition_mask = values <= threshold

                elif condition == "POSITIVE":
                    condition_mask = values > 0

                elif condition == "NEGATIVE":
                    condition_mask = values < 0

                else:
                    raise ValueError(
                        f"Unknown condition: {condition}"
                    )

                for quarter in np.unique(selected_quarters):

                    qmask = selected_quarters == quarter

                    cmask = qmask & condition_mask

                    s = stats[(feature, quarter)]

                    s["Valid_N"] += int(qmask.sum())

                    s["Valid_Events"] += int(
                        selected_events[qmask].sum()
                    )

                    s["Condition_N"] += int(cmask.sum())

                    s["Condition_Events"] += int(
                        selected_events[cmask].sum()
                    )

            row_offset = batch_end

        assert row_offset == pf.metadata.num_rows

        print("Completed.")

    # --------------------------------------------------------
    # 5. Build quarterly results
    # --------------------------------------------------------

    output_rows = []

    for (feature, quarter), s in stats.items():

        valid_n = s["Valid_N"]

        condition_n = s["Condition_N"]

        valid_rate = (
            s["Valid_Events"] / valid_n
            if valid_n else np.nan
        )

        event_probability = (
            s["Condition_Events"] / condition_n
            if condition_n else np.nan
        )

        lift = (
            event_probability / valid_rate
            if valid_rate > 0 and condition_n > 0
            else np.nan
        )

        output_rows.append({
            "Feature": feature,
            "Parent_Sensor": candidate_info.loc[
                feature, "Parent_Sensor"
            ],
            "Quarter": quarter,
            "Condition": candidate_info.loc[
                feature, "Condition"
            ],
            "Coverage": valid_n / quarter_totals[quarter],
            "Valid_N": valid_n,
            "Valid_Events": s["Valid_Events"],
            "Condition_N": condition_n,
            "Condition_Events": s["Condition_Events"],
            "Valid_Event_Rate": valid_rate,
            "Event_Probability": event_probability,
            "Lift": lift
        })

    results = pd.DataFrame(output_rows)

    # --------------------------------------------------------
    # 6. Save
    # --------------------------------------------------------

    output_path = os.path.join(
        SCREENING_DIR,
        output_filename
    )

    results.to_csv(output_path, index=False)

    print("\nQUARTERLY STABILITY ANALYSIS COMPLETED")

    print("Features analyzed:", len(feature_names))

    print("Result rows:", len(results))

    print("Saved:", output_path)

    return results


print("Quarterly stability function is ready.")

Analyze the Five Alternatives

In [ ]:
# ============================================================
# STEP 11F.1: RUN ALTERNATIVE CANDIDATE STABILITY ANALYSIS
# ============================================================

alternative_quarterly = calculate_quarterly_stability(
    selected_candidates=alternative_candidates,
    output_filename="alternative5_quarterly_stability.csv"
)

print("\nQUARTERLY LIFT COMPARISON")
print("=" * 65)

lift_pivot = alternative_quarterly.pivot(
    index="Quarter",
    columns="Parent_Sensor",
    values="Lift"
)

display(
    lift_pivot.round(3)
)

print("\nQUARTERLY SUPPORT DETAILS")
print("=" * 65)

display(
    alternative_quarterly[
        [
            "Parent_Sensor",
            "Quarter",
            "Coverage",
            "Condition_N",
            "Condition_Events",
            "Valid_Event_Rate",
            "Event_Probability",
            "Lift"
        ]
    ].round(4)
)

Episode Support for the Five Alternative Features

Alternative Candidate Episode Support

In [ ]:
# ============================================================
# STEP 11G: EPISODE SUPPORT — FIVE ALTERNATIVE FEATURES
# ============================================================

import os
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

# ------------------------------------------------------------
# 1. Load candidate definitions and episode mapping
# ------------------------------------------------------------

alt_path = os.path.join(
    SCREENING_DIR, "alternative_5_candidates.csv"
)

map_path = os.path.join(
    SCREENING_DIR, "discovery_episode_mapping.parquet"
)

alt_candidates = pd.read_csv(alt_path)
episode_map = pd.read_parquet(map_path)

features = alt_candidates["Feature"].tolist()
info = alt_candidates.set_index("Feature")

assert len(features) == 5
assert len(set(features)) == 5

# ------------------------------------------------------------
# 2. Prepare source-row alignment
# ------------------------------------------------------------

lookups = {}

for name in ["NON_EXTREME", "EXTREME"]:

    subset = (
        episode_map.loc[
            episode_map["source_dataset"] == name
        ]
        .sort_values("source_row")
    )

    lookups[name] = {
        "rows": subset["source_row"].to_numpy(np.int64),
        "event": subset["event"].to_numpy(np.uint8),
        "run": subset["Positive_Run_ID"].to_numpy(np.int64),
        "block": subset["Block_48h"].to_numpy(np.int32)
    }

n_blocks = int(episode_map["Block_48h"].max()) + 1

# ------------------------------------------------------------
# 3. Initialize counters
# ------------------------------------------------------------

support = {}

for feature in features:

    support[feature] = {
        "condition_n": 0,
        "event_n": 0,
        "runs": set(),
        "active_blocks": np.zeros(n_blocks, dtype=np.int64),
        "positive_blocks": np.zeros(n_blocks, dtype=np.int64)
    }

# ------------------------------------------------------------
# 4. Read selected columns in PyArrow batches
# ------------------------------------------------------------

columns = []

for feature in features:
    columns.extend([
        feature,
        validity_mapping[feature]
    ])

for name, path in [
    ("NON_EXTREME", normal_file),
    ("EXTREME", extreme_file)
]:

    print(f"Processing {name}...")

    pf = pq.ParquetFile(path)
    lookup = lookups[name]

    offset = 0

    for batch in pf.iter_batches(
        batch_size=32768,
        columns=columns,
        use_threads=True
    ):

        end = offset + batch.num_rows

        left = np.searchsorted(lookup["rows"], offset)
        right = np.searchsorted(lookup["rows"], end)

        if right == left:
            offset = end
            continue

        positions = lookup["rows"][left:right] - offset

        events = lookup["event"][left:right]
        runs = lookup["run"][left:right]
        blocks = lookup["block"][left:right]

        for i, feature in enumerate(features):

            flags = pc.fill_null(
                batch.column(2 * i + 1),
                False
            ).to_numpy(zero_copy_only=False)

            valid = flags[positions]

            if not valid.any():
                continue

            selected_positions = positions[valid]

            values = pc.cast(
                pc.take(
                    batch.column(2 * i),
                    pa.array(selected_positions)
                ),
                pa.float64(),
                safe=False
            ).to_numpy(zero_copy_only=False)

            selected_events = events[valid]
            selected_runs = runs[valid]
            selected_blocks = blocks[valid]

            finite = np.isfinite(values)

            values = values[finite]
            selected_events = selected_events[finite]
            selected_runs = selected_runs[finite]
            selected_blocks = selected_blocks[finite]

            condition = info.loc[feature, "Condition"]
            threshold = float(info.loc[feature, "Threshold"])

            if condition == "HIGH_90":
                mask = values >= threshold
            elif condition == "LOW_10":
                mask = values <= threshold
            elif condition == "POSITIVE":
                mask = values > 0
            elif condition == "NEGATIVE":
                mask = values < 0
            else:
                raise ValueError(condition)

            s = support[feature]

            s["condition_n"] += int(mask.sum())

            active_blocks = selected_blocks[mask]

            s["active_blocks"] += np.bincount(
                active_blocks,
                minlength=n_blocks
            )

            positive = mask & (selected_events == 1)

            s["event_n"] += int(positive.sum())

            s["runs"].update(
                np.unique(
                    selected_runs[positive]
                ).tolist()
            )

            s["positive_blocks"] += np.bincount(
                selected_blocks[positive],
                minlength=n_blocks
            )

        offset = end

    assert offset == pf.metadata.num_rows

    print("Completed.")

# ------------------------------------------------------------
# 5. Calculate support statistics
# ------------------------------------------------------------

rows = []

for feature in features:

    s = support[feature]

    active_count = int(
        np.count_nonzero(s["active_blocks"])
    )

    positive_count = int(
        np.count_nonzero(s["positive_blocks"])
    )

    largest = int(s["positive_blocks"].max())

    rows.append({
        "Feature": feature,
        "Parent_Sensor": info.loc[feature, "Parent_Sensor"],
        "Condition": info.loc[feature, "Condition"],
        "Condition_N": s["condition_n"],
        "Condition_Events": s["event_n"],
        "Positive_Runs": len(s["runs"] - {0}),
        "Active_48h_Blocks": active_count,
        "Positive_48h_Blocks": positive_count,
        "Positive_Block_Fraction": (
            positive_count / active_count
            if active_count else np.nan
        ),
        "Largest_Block_Share": (
            largest / s["event_n"]
            if s["event_n"] else np.nan
        )
    })

alternative_support = pd.DataFrame(rows)

# ------------------------------------------------------------
# 6. Verify against original screening
# ------------------------------------------------------------

for _, row in alternative_support.iterrows():

    original = info.loc[row["Feature"]]

    assert row["Condition_N"] == int(
        original["Condition_Observations"]
    )

    assert row["Condition_Events"] == int(
        original["Condition_Events"]
    )

print("\nSCREENING COUNT CONSISTENCY: PASSED")

# ------------------------------------------------------------
# 7. Save and display
# ------------------------------------------------------------

output_path = os.path.join(
    SCREENING_DIR,
    "alternative5_episode_support.csv"
)

alternative_support.to_csv(output_path, index=False)

print("\nALTERNATIVE CANDIDATE EPISODE SUPPORT")
print("=" * 70)

display(alternative_support.round(4))

print("\nSaved:", output_path)

Compare Feature Conditions Within the Same Quarter

Matched-Quarter Candidate Comparison

In [ ]:
# ============================================================
# STEP 12A: WITHIN-QUARTER CANDIDATE COMPARISON
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Load existing quarterly results
# ------------------------------------------------------------

original_quarterly = pd.read_csv(
    os.path.join(
        SCREENING_DIR,
        "top5_quarterly_stability.csv"
    )
)

alternative_quarterly = pd.read_csv(
    os.path.join(
        SCREENING_DIR,
        "alternative5_quarterly_stability.csv"
    )
)

all_quarterly = pd.concat(
    [original_quarterly, alternative_quarterly],
    ignore_index=True
)

# ------------------------------------------------------------
# 2. Select our three provisional candidates
# ------------------------------------------------------------

shortlisted_sensors = [
    "funding_range",
    "oi_extreme_recency",
    "funding_change"
]

matched_df = all_quarterly.loc[
    all_quarterly["Parent_Sensor"].isin(shortlisted_sensors)
].copy()

assert matched_df["Feature"].nunique() == 3

assert not matched_df.duplicated(
    ["Feature", "Quarter"]
).any()

# ------------------------------------------------------------
# 3. Calculate condition-false observations
# ------------------------------------------------------------

matched_df["Other_N"] = (
    matched_df["Valid_N"]
    - matched_df["Condition_N"]
)

matched_df["Other_Events"] = (
    matched_df["Valid_Events"]
    - matched_df["Condition_Events"]
)

assert (matched_df["Other_N"] >= 0).all()
assert (matched_df["Other_Events"] >= 0).all()
assert (
    matched_df["Other_Events"] <= matched_df["Other_N"]
).all()

# ------------------------------------------------------------
# 4. Calculate within-quarter event probabilities
# ------------------------------------------------------------

matched_df["Condition_Rate"] = np.where(
    matched_df["Condition_N"] > 0,
    matched_df["Condition_Events"]
    / matched_df["Condition_N"].replace(0, np.nan),
    np.nan
)

matched_df["Other_Rate"] = np.where(
    matched_df["Other_N"] > 0,
    matched_df["Other_Events"]
    / matched_df["Other_N"].replace(0, np.nan),
    np.nan
)

matched_df["Within_Quarter_RR"] = (
    matched_df["Condition_Rate"]
    / matched_df["Other_Rate"].replace(0, np.nan)
)

matched_df["Risk_Difference_pp"] = (
    matched_df["Condition_Rate"]
    - matched_df["Other_Rate"]
) * 100

# ------------------------------------------------------------
# 5. Build quarter-adjusted summaries
# ------------------------------------------------------------

summary_rows = []

for feature, group in matched_df.groupby("Feature"):

    # Only quarters with both comparison groups
    eligible = group.loc[
        (group["Condition_N"] > 0)
        &
        (group["Other_N"] > 0)
        &
        group["Other_Rate"].notna()
    ].copy()

    # Expected positive events if condition observations
    # had the event rate of other valid observations
    # in their own quarter.
    expected_events = (
        eligible["Condition_N"]
        * eligible["Other_Rate"]
    ).sum()

    observed_events = eligible["Condition_Events"].sum()

    condition_n = eligible["Condition_N"].sum()

    observed_rate = (
        observed_events / condition_n
        if condition_n else np.nan
    )

    expected_rate = (
        expected_events / condition_n
        if condition_n else np.nan
    )

    adjusted_rr = (
        observed_rate / expected_rate
        if expected_rate > 0
        else np.nan
    )

    summary_rows.append({
        "Feature": feature,
        "Parent_Sensor": group["Parent_Sensor"].iloc[0],
        "Comparable_Quarters": len(eligible),
        "Quarters_RR_Above_1": int(
            (eligible["Within_Quarter_RR"] > 1).sum()
        ),
        "Condition_N": int(condition_n),
        "Observed_Events": int(observed_events),
        "Observed_Event_Rate": observed_rate,
        "Quarter_Matched_Other_Rate": expected_rate,
        "Quarter_Adjusted_RR": adjusted_rr,
        "Risk_Difference_pp": (
            (observed_rate - expected_rate) * 100
        )
    })

matched_summary = pd.DataFrame(summary_rows)

# ------------------------------------------------------------
# 6. Display quarterly comparisons
# ------------------------------------------------------------

print("WITHIN-QUARTER COMPARISON")
print("=" * 70)

display(
    matched_df[
        [
            "Parent_Sensor",
            "Quarter",
            "Condition_N",
            "Condition_Events",
            "Other_N",
            "Other_Events",
            "Condition_Rate",
            "Other_Rate",
            "Within_Quarter_RR",
            "Risk_Difference_pp"
        ]
    ].round(4)
)

print("\nQUARTER-ADJUSTED CANDIDATE SUMMARY")
print("=" * 70)

display(
    matched_summary.sort_values(
        "Quarter_Adjusted_RR",
        ascending=False
    ).round(4)
)

# ------------------------------------------------------------
# 7. Save results
# ------------------------------------------------------------

matched_df.to_csv(
    os.path.join(
        SCREENING_DIR,
        "top3_within_quarter_comparison.csv"
    ),
    index=False
)

matched_summary.to_csv(
    os.path.join(
        SCREENING_DIR,
        "top3_quarter_adjusted_summary.csv"
    ),
    index=False
)

print("\nMatched-quarter results saved.")

Verify Parent Sensors and Final Research Definitions

Prepare Parent-Feature Verification

In [ ]:
# ============================================================
# STEP 12B: PREPARE FINAL FEATURE DEFINITION REVIEW
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# 1. Select provisional top three
# ------------------------------------------------------------

final_feature_names = [
    "f4z__funding__funding_range_percentile__1h__value",
    "f4z__oi__oi_extreme_recency_percentile__1h__value",
    "f4z__funding__funding_change_rz__4h__value"
]

# ------------------------------------------------------------
# 2. Retrieve feature metadata
# ------------------------------------------------------------

final_metadata = feature_catalog_v2.loc[
    feature_catalog_v2["Feature"].isin(final_feature_names)
].copy()

# Preserve intended ranking
final_metadata["Rank"] = final_metadata["Feature"].map(
    {
        name: i + 1
        for i, name in enumerate(final_feature_names)
    }
)

final_metadata = final_metadata.sort_values("Rank")

assert len(final_metadata) == 3

# ------------------------------------------------------------
# 3. Display complete metadata
# ------------------------------------------------------------

pd.set_option("display.max_colwidth", None)

print("FINAL FEATURE METADATA REVIEW")
print("=" * 70)

display(
    final_metadata[
        [
            "Rank",
            "Feature",
            "Family",
            "Parent_Sensor",
            "Transformation",
            "Lookback_Count",
            "Lookback_1",
            "Lookback_2"
        ]
    ]
)

# ------------------------------------------------------------
# 4. Save verification table
# ------------------------------------------------------------

output_path = os.path.join(
    SCREENING_DIR,
    "top3_feature_metadata.csv"
)

final_metadata.to_csv(
    output_path,
    index=False
)

print("\nFeature metadata saved:")
print(output_path)

Identify the Underlying Parent Feature Definitions

Search for Parent Feature Definitions

In [ ]:
# ============================================================
# STEP 12C: LOCATE FEATURE DEFINITION REFERENCES
# ============================================================

import os

print("SEARCHING PROJECT REFERENCE FILES")
print("=" * 65)

search_roots = [
    os.path.join(PROJECT_DIR, "References"),
    os.path.join(PROJECT_DIR, "Results")
]

reference_keywords = [
    "dictionary",
    "formula",
    "sensor",
    "feature",
    "reference",
    "contract"
]

matching_files = []

for root in search_roots:

    if not os.path.isdir(root):
        continue

    for current_dir, folders, files in os.walk(root):

        # Avoid searching output-heavy screening folders
        folders[:] = [
            folder for folder in folders
            if folder != "02_F4Z_Feature_Screening"
        ]

        for filename in files:

            if any(
                keyword in filename.lower()
                for keyword in reference_keywords
            ):

                full_path = os.path.join(
                    current_dir,
                    filename
                )

                matching_files.append(full_path)

print("Matching reference files:", len(matching_files))

for path in matching_files[:50]:
    print(path)

if not matching_files:
    print(
        "\nNo matching reference files found "
        "in the searched project folders."
    )

Locate Reference Files Inside the X Archive

In [ ]:
# ============================================================
# STEP 12D: LOCATE ORIGINAL FORGE REFERENCE DOCUMENTATION
# ============================================================

import os
import zipfile

print("SEARCHING GOOGLE DRIVE FOR FORGE X ARCHIVE")
print("=" * 65)

drive_root = "/content/drive/MyDrive"

zip_files = []

# Search names only; do not extract archives
for current_dir, folders, files in os.walk(drive_root):

    for filename in files:

        if filename.lower().endswith(".zip"):

            full_path = os.path.join(
                current_dir,
                filename
            )

            zip_files.append(full_path)

print("\nZIP archives found:", len(zip_files))

for i, path in enumerate(zip_files):
    print(f"{i + 1}. {path}")

# ------------------------------------------------------------
# Inspect archives for FORGE reference documentation
# ------------------------------------------------------------

print("\nSEARCHING ARCHIVE CONTENTS")
print("=" * 65)

reference_files = []

for archive_path in zip_files:

    try:
        with zipfile.ZipFile(archive_path, "r") as z:

            matches = [
                name for name in z.namelist()
                if name.lower().endswith(".json")
                and (
                    "/reference/" in name.lower()
                    or "dictionary" in name.lower()
                    or "formula" in name.lower()
                )
            ]

            if matches:

                print("\nARCHIVE:")
                print(archive_path)

                for name in matches:

                    print("  ", name)

                    reference_files.append({
                        "Archive": archive_path,
                        "Reference_File": name
                    })

    except (zipfile.BadZipFile, OSError) as e:

        print("Could not inspect:", archive_path)
        print("Reason:", str(e))

print("\nTotal reference files located:", len(reference_files))

Extract the Three Feature Dictionary Entries



In [ ]:
# ============================================================
# STEP 12E: INSPECT THREE FEATURES IN AUTHORITATIVE DICTIONARY
# ============================================================

import os
import json
import zipfile
import pandas as pd

# ------------------------------------------------------------
# 1. Locate Y archive and dictionary
# ------------------------------------------------------------

y_zip_path = os.path.join(
    DATA_DIR,
    "HANDOFF_2_CANONICAL_XY_V1_1.zip"
)

dictionary_path = (
    "HANDOFF_2_CANONICAL_XY_V1_1/"
    "FEATURE_DICTIONARY.json"
)

# ------------------------------------------------------------
# 2. Read feature dictionary without extraction
# ------------------------------------------------------------

with zipfile.ZipFile(y_zip_path, "r") as z:

    with z.open(dictionary_path) as f:
        feature_dictionary = json.load(f)

print("FEATURE DICTIONARY")
print("=" * 65)

print("Data type:", type(feature_dictionary).__name__)
print("Total entries:", len(feature_dictionary))

# ------------------------------------------------------------
# 3. Locate the three exact F4Z columns
# ------------------------------------------------------------

selected_features = [
    "f4z__funding__funding_range_percentile__1h__value",
    "f4z__oi__oi_extreme_recency_percentile__1h__value",
    "f4z__funding__funding_change_rz__4h__value"
]

# Handle both list and dictionary-based structures
if isinstance(feature_dictionary, list):

    dictionary_entries = feature_dictionary

elif isinstance(feature_dictionary, dict):

    dictionary_entries = [
        {"column": key, **value}
        for key, value in feature_dictionary.items()
        if isinstance(value, dict)
    ]

else:
    raise TypeError("Unexpected dictionary structure")

# ------------------------------------------------------------
# 4. Find matching entries
# ------------------------------------------------------------

matched_entries = {}

for feature in selected_features:

    matches = [
        entry
        for entry in dictionary_entries
        if entry.get("column", "").lower() == feature.lower()
    ]

    matched_entries[feature] = matches

    print("\n" + "=" * 70)
    print("FEATURE:", feature)
    print("=" * 70)

    if not matches:
        print("NOT FOUND IN DICTIONARY")
        continue

    for entry in matches:

        print(
            json.dumps(
                entry,
                indent=2,
                default=str
            )
        )

# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

print("\nDICTIONARY MATCH SUMMARY")
print("=" * 65)

for feature, matches in matched_entries.items():

    print(
        feature,
        "→",
        len(matches),
        "matching entries"
    )

Verify Formula-Document Consistency

In [ ]:
# ============================================================
# STEP 12F: COMPARE X AND Y SENSOR FORMULA DOCUMENTS
# ============================================================

import os
import zipfile
import hashlib
import json

# ------------------------------------------------------------
# 1. Define archive paths
# ------------------------------------------------------------

x_zip_path = os.path.join(
    DATA_DIR,
    "Forge_Complete_90pct_2022-04-22_to_2025-01-23.zip"
)

y_zip_path = os.path.join(
    DATA_DIR,
    "HANDOFF_2_CANONICAL_XY_V1_1.zip"
)

x_formula_path = "reference/F4_SENSOR_FORMULAS.json"

y_formula_path = (
    "HANDOFF_2_CANONICAL_XY_V1_1/"
    "reference/F4_SENSOR_FORMULAS.json"
)

# ------------------------------------------------------------
# 2. Read only the formula JSON files
# ------------------------------------------------------------

with zipfile.ZipFile(x_zip_path, "r") as z:
    x_bytes = z.read(x_formula_path)

with zipfile.ZipFile(y_zip_path, "r") as z:
    y_bytes = z.read(y_formula_path)

# ------------------------------------------------------------
# 3. Calculate SHA-256 hashes
# ------------------------------------------------------------

x_hash = hashlib.sha256(x_bytes).hexdigest()
y_hash = hashlib.sha256(y_bytes).hexdigest()

print("F4 SENSOR FORMULA DOCUMENT VERIFICATION")
print("=" * 70)

print("X formula SHA-256:", x_hash)
print("Y formula SHA-256:", y_hash)

print("\nByte-for-byte identical:", x_hash == y_hash)

# ------------------------------------------------------------
# 4. Compare parsed JSON structures
# ------------------------------------------------------------

x_formulas = json.loads(x_bytes)
y_formulas = json.loads(y_bytes)

print("Parsed JSON identical:", x_formulas == y_formulas)

print("\nX document type:", type(x_formulas).__name__)
print("Y document type:", type(y_formulas).__name__)

print("\nX top-level entries:", len(x_formulas))
print("Y top-level entries:", len(y_formulas))

# ------------------------------------------------------------
# 5. Display top-level structure only
# ------------------------------------------------------------

if isinstance(x_formulas, dict):

    print("\nX top-level keys:")
    print(list(x_formulas.keys())[:30])

elif isinstance(x_formulas, list):

    print("\nFirst X formula entry:")
    print(
        json.dumps(
            x_formulas[0],
            indent=2,
            default=str
        )[:2500]
    )

Verify Formula Hashes and Parent Sensor IDs

In [ ]:
# ============================================================
# STEP 12G: VERIFY THREE PARENT SENSOR FORMULAS
# ============================================================

import json
import pandas as pd

# ------------------------------------------------------------
# 1. Selected features and expected sensors
# ------------------------------------------------------------

selected_sensors = [
    "funding_range",
    "oi_extreme_recency",
    "funding_change"
]

# feature_dictionary and x_formulas were loaded previously.

# ------------------------------------------------------------
# 2. Build dictionary lookup
# ------------------------------------------------------------

dictionary_lookup = {
    entry["sensor"]: entry
    for entry in feature_dictionary
    if (
        entry.get("sensor") in selected_sensors
        and entry.get("layer") == "F4Z"
        and entry.get("column", "").lower() in {
            "f4z__funding__funding_range_percentile__1h__value",
            "f4z__oi__oi_extreme_recency_percentile__1h__value",
            "f4z__funding__funding_change_rz__4h__value"
        }
    )
}

# ------------------------------------------------------------
# 3. Match F4 sensor definitions
# ------------------------------------------------------------

verification_rows = []

for sensor in selected_sensors:

    dictionary_entry = dictionary_lookup.get(sensor)

    formula_matches = [
        entry for entry in x_formulas
        if entry.get("name") == sensor
    ]

    print("\n" + "=" * 70)
    print("PARENT SENSOR:", sensor)
    print("=" * 70)

    if dictionary_entry is None:
        print("Dictionary entry not found.")
        continue

    if len(formula_matches) != 1:
        print(
            "Expected one F4 formula, found:",
            len(formula_matches)
        )
        continue

    formula_entry = formula_matches[0]

    dictionary_hash = dictionary_entry.get(
        "formula_definition_sha256"
    )

    formula_hash = formula_entry.get(
        "definition_sha256"
    )

    hash_match = dictionary_hash == formula_hash

    print("Sensor ID:", formula_entry.get("sensor_id"))
    print("Formula:", formula_entry.get("formula"))
    print("Dictionary hash:", dictionary_hash)
    print("F4 formula hash:", formula_hash)
    print("Hash match:", hash_match)

    print("\nParent inputs:")
    print(formula_entry.get("inputs"))

    verification_rows.append({
        "Parent_Sensor": sensor,
        "Sensor_ID": formula_entry.get("sensor_id"),
        "Dictionary_Formula": dictionary_entry.get("formula"),
        "F4_Formula": formula_entry.get("formula"),
        "Dictionary_Hash": dictionary_hash,
        "F4_Hash": formula_hash,
        "Hash_Match": hash_match,
        "Orientation": formula_entry.get("orientation"),
        "Parent_Inputs": formula_entry.get("inputs")
    })

# ------------------------------------------------------------
# 4. Display verification summary
# ------------------------------------------------------------

sensor_verification = pd.DataFrame(verification_rows)

print("\nPARENT SENSOR VERIFICATION SUMMARY")
print("=" * 70)

display(
    sensor_verification[
        [
            "Parent_Sensor",
            "Sensor_ID",
            "Hash_Match",
            "Dictionary_Formula",
            "F4_Formula"
        ]
    ]
)

if len(sensor_verification) == 3:

    print(
        "\nAll three hashes match:",
        sensor_verification["Hash_Match"].all()
    )

else:

    print(
        "\nWARNING: Not all three sensors were matched."
    )

# ------------------------------------------------------------
# 5. Save verification
# ------------------------------------------------------------

output_path = os.path.join(
    SCREENING_DIR,
    "top3_parent_sensor_verification.csv"
)

sensor_verification.to_csv(
    output_path,
    index=False
)

print("\nSaved:", output_path)

Inspect Target-Related Parquet Metadata

In [ ]:
# ============================================================
# STEP 12H: VERIFY TARGET COLUMN AND PARQUET METADATA
# ============================================================

import pyarrow.parquet as pq

print("TARGET DEFINITION INSPECTION")
print("=" * 70)

target_keywords = [
    "return",
    "horizon",
    "target",
    "label",
    "price",
    "decision",
    "future",
    "bps"
]

for source_name, file_path in [
    ("NON_EXTREME", normal_file),
    ("EXTREME", extreme_file)
]:

    print("\n" + "=" * 70)
    print("DATASET:", source_name)
    print("=" * 70)

    pf = pq.ParquetFile(file_path)

    print("Rows:", pf.metadata.num_rows)
    print("Columns:", pf.metadata.num_columns)

    # --------------------------------------------------------
    # Relevant schema fields
    # --------------------------------------------------------

    print("\nTARGET-RELATED COLUMNS:")

    matching_fields = [
        field
        for field in pf.schema_arrow
        if any(
            keyword in field.name.lower()
            for keyword in target_keywords
        )
        and not field.name.lower().startswith("f4")
    ]

    for field in matching_fields[:70]:
        print(f"{field.name}: {field.type}")

    # --------------------------------------------------------
    # File-level metadata
    # --------------------------------------------------------

    print("\nFILE METADATA:")

    metadata = pf.schema_arrow.metadata

    if metadata:

        for key, value in metadata.items():

            key_text = key.decode(
                "utf-8", errors="replace"
            )

            value_text = value.decode(
                "utf-8", errors="replace"
            )

            print(
                key_text,
                ":",
                value_text[:1000]
            )

    else:

        print("No custom Arrow schema metadata found.")

Inspect Actual Target Values

In [ ]:
# ============================================================
# STEP 12I: INSPECT TARGET VALUES AND HORIZON
# ============================================================

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

target_columns = [
    "decision_time",
    "horizon_minutes",
    "label_status",
    "signed_return",
    "net_magnitude_log_bps"
]

for source_name, file_path in [
    ("NON_EXTREME", normal_file),
    ("EXTREME", extreme_file)
]:

    print("\n" + "=" * 70)
    print("DATASET:", source_name)
    print("=" * 70)

    pf = pq.ParquetFile(file_path)

    horizon_counts = {}
    status_counts = {}

    sample_batches = []
    sample_size = 10

    for batch in pf.iter_batches(
        batch_size=32768,
        columns=target_columns,
        use_threads=True
    ):

        # Small batch only
        df = batch.to_pandas()

        # Count horizon values
        for value, count in df[
            "horizon_minutes"
        ].value_counts(dropna=False).items():

            key = str(value)

            horizon_counts[key] = (
                horizon_counts.get(key, 0) + int(count)
            )

        # Count label status values
        for value, count in df[
            "label_status"
        ].value_counts(dropna=False).items():

            key = str(value)

            status_counts[key] = (
                status_counts.get(key, 0) + int(count)
            )

        # Keep only first ten sample rows
        if sum(len(x) for x in sample_batches) < sample_size:

            remaining = sample_size - sum(
                len(x) for x in sample_batches
            )

            sample_batches.append(
                df.head(remaining)
            )

    print("\nHORIZON VALUES:")
    print(horizon_counts)

    print("\nLABEL STATUS VALUES:")
    print(status_counts)

    sample_df = pd.concat(
        sample_batches,
        ignore_index=True
    )

    print("\nSAMPLE TARGET ROWS:")

    display(
        sample_df
    )

    # --------------------------------------------------------
    # Compare signed_return with net magnitude
    # --------------------------------------------------------

    sample_df["Absolute_Signed_Return"] = (
        sample_df["signed_return"].abs()
    )

    sample_df["Magnitude_Difference"] = (
        sample_df["Absolute_Signed_Return"]
        - sample_df["net_magnitude_log_bps"]
    )

    print("\nSAMPLE RETURN COMPARISON:")

    display(
        sample_df[
            [
                "signed_return",
                "net_magnitude_log_bps",
                "Absolute_Signed_Return",
                "Magnitude_Difference"
            ]
        ].round(6)
    )

Confirm the Return Calculation

Locate the Authoritative Y-Label Formula

In [ ]:
# ============================================================
# STEP 12J: LOCATE Y-LABEL DEFINITION FILES
# ============================================================

import zipfile
import os

y_zip_path = os.path.join(
    DATA_DIR,
    "HANDOFF_2_CANONICAL_XY_V1_1.zip"
)

keywords = [
    "label",
    "target",
    "return",
    "build_y",
    "readme",
    "contract",
    "manifest",
    "formula"
]

print("SEARCHING Y ARCHIVE FOR LABEL DOCUMENTATION")
print("=" * 70)

with zipfile.ZipFile(y_zip_path, "r") as z:

    matches = [
        name
        for name in z.namelist()
        if any(
            keyword in name.lower()
            for keyword in keywords
        )
        and not name.endswith("/")
    ]

print("Matching files:", len(matches))

for name in matches[:100]:
    print(name)

if len(matches) > 100:
    print(
        f"\nAdditional matching files not displayed: "
        f"{len(matches) - 100}"
    )

List Documentation and Target-Building Scripts

In [ ]:
# ============================================================
# STEP 12K: FIND TARGET DOCUMENTATION AND BUILD SCRIPTS
# ============================================================

import zipfile
import os

y_zip_path = os.path.join(
    DATA_DIR,
    "HANDOFF_2_CANONICAL_XY_V1_1.zip"
)

print("Y ARCHIVE — DOCUMENTATION AND SCRIPT FILES")
print("=" * 70)

with zipfile.ZipFile(y_zip_path, "r") as z:

    files = [
        name for name in z.namelist()
        if not name.endswith("/")
    ]

# Exclude Parquet and other data files
documentation_files = [
    name for name in files
    if name.lower().endswith(
        (".json", ".txt", ".md", ".py", ".yaml", ".yml")
    )
]

print("Documentation/script files:", len(documentation_files))

for name in documentation_files:
    print(name)

# ------------------------------------------------------------
# Identify especially relevant files
# ------------------------------------------------------------

keywords = [
    "return",
    "label",
    "target",
    "build",
    "manifest",
    "readme",
    "contract"
]

relevant_files = [
    name for name in documentation_files
    if any(
        keyword in os.path.basename(name).lower()
        for keyword in keywords
    )
]

print("\n" + "=" * 70)
print("POTENTIALLY RELEVANT TARGET DOCUMENTATION")
print("=" * 70)

for name in relevant_files:
    print(name)

Read the Return Schema and Dataset Specification

In [ ]:
# ============================================================
# STEP 12L: READ OFFICIAL RETURN DOCUMENTATION
# ============================================================

import os
import json
import zipfile

y_zip_path = os.path.join(
    DATA_DIR,
    "HANDOFF_2_CANONICAL_XY_V1_1.zip"
)

root = "HANDOFF_2_CANONICAL_XY_V1_1/"

files_to_read = [
    root + "team_tools/return_schema.json",
    root + "team_tools/TEAM_DATASET_SPEC.md"
]

print("OFFICIAL RETURN DOCUMENTATION")
print("=" * 70)

with zipfile.ZipFile(y_zip_path, "r") as z:

    for file_path in files_to_read:

        print("\n" + "=" * 70)
        print("FILE:", file_path)
        print("=" * 70)

        content = z.read(file_path).decode(
            "utf-8",
            errors="replace"
        )

        if file_path.endswith(".json"):

            parsed = json.loads(content)

            print(
                json.dumps(
                    parsed,
                    indent=2,
                    default=str
                )
            )

        else:

            # Display first 15,000 characters
            print(content[:15000])

            if len(content) > 15000:
                print(
                    "\n[Additional content not displayed]"
                )

Inspect the Screening Dataset's Provenance

Screening Dataset Structure

In [ ]:
# ============================================================
# STEP 12M: SCREENING DATASET STRUCTURE AND PROVENANCE
# ============================================================

import pyarrow.parquet as pq

print("SCREENING DATASET PROVENANCE INSPECTION")
print("=" * 70)

keywords = [
    "source",
    "version",
    "build",
    "release",
    "symbol",
    "binding",
    "status",
    "available",
    "price"
]

for source_name, path in [
    ("NON_EXTREME", normal_file),
    ("EXTREME", extreme_file)
]:

    print("\n" + "=" * 70)
    print("DATASET:", source_name)
    print("=" * 70)

    pf = pq.ParquetFile(path)

    fields = [
        field.name
        for field in pf.schema_arrow
    ]

    # Only inspect non-feature columns
    metadata_fields = [
        name for name in fields
        if not name.lower().startswith(("f4__", "f4z__"))
    ]

    print("\nNON-F4/F4Z COLUMNS:")

    for name in metadata_fields:
        print(name)

    print("\nPOTENTIAL PROVENANCE COLUMNS:")

    provenance_fields = [
        name for name in metadata_fields
        if any(
            keyword in name.lower()
            for keyword in keywords
        )
    ]

    for name in provenance_fields:
        print(name)

    print("\nTOTAL COLUMNS:", len(fields))
    print("NON-FEATURE COLUMNS:", len(metadata_fields))

Verify Screening Dataset Consistency

Final Screening Dataset Consistency Audit

In [ ]:
# ============================================================
# STEP 12N: SCREENING DATASET CONSISTENCY AUDIT
# ============================================================

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

columns = [
    "canonical_symbol",
    "direction",
    "net_magnitude_log_bps",
    "signed_return",
    "label_status"
]

audit_rows = []

for source_name, file_path in [
    ("NON_EXTREME", normal_file),
    ("EXTREME", extreme_file)
]:

    print(f"\nAuditing {source_name}...")

    pf = pq.ParquetFile(file_path)

    total_rows = 0
    invalid_labels = 0
    return_mismatches = 0
    partition_violations = 0
    symbols = set()

    for batch in pf.iter_batches(
        batch_size=65536,
        columns=columns,
        use_threads=True
    ):

        df = batch.to_pandas()

        total_rows += len(df)

        symbols.update(
            df["canonical_symbol"].dropna().unique()
        )

        invalid_labels += int(
            (df["label_status"] != "VALID").sum()
        )

        expected_return = (
            df["direction"]
            * df["net_magnitude_log_bps"]
        )

        return_mismatches += int(
            (~np.isclose(
                df["signed_return"],
                expected_return,
                rtol=1e-10,
                atol=1e-8,
                equal_nan=False
            )).sum()
        )

        if source_name == "NON_EXTREME":

            violations = (
                (df["signed_return"] < -500)
                |
                (df["signed_return"] > 500)
            )

        else:

            violations = (
                (df["signed_return"] >= -500)
                &
                (df["signed_return"] <= 500)
            )

        partition_violations += int(
            violations.sum()
        )

    audit_rows.append({
        "Dataset": source_name,
        "Rows": total_rows,
        "Symbols": ", ".join(sorted(symbols)),
        "Invalid_Labels": invalid_labels,
        "Return_Mismatches": return_mismatches,
        "Partition_Violations": partition_violations
    })

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

audit_df = pd.DataFrame(audit_rows)

print("\nFINAL SCREENING DATASET AUDIT")
print("=" * 70)

display(audit_df)

# ------------------------------------------------------------
# Save audit
# ------------------------------------------------------------

audit_path = os.path.join(
    SCREENING_DIR,
    "screening_dataset_consistency_audit.csv"
)

audit_df.to_csv(audit_path, index=False)

print("\nAudit saved:")
print(audit_path)

Consolidated Candidate Comparison

In [ ]:
# ============================================================
# STEP 13A: COMBINE EVIDENCE FOR 10 F4Z CANDIDATES
# ============================================================

import os
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Load candidate rankings
# ------------------------------------------------------------

ranking = pd.read_csv(
    os.path.join(
        SCREENING_DIR,
        "best_condition_per_parent.csv"
    )
)

# ------------------------------------------------------------
# 2. Load quarterly results
# ------------------------------------------------------------

q1 = pd.read_csv(
    os.path.join(
        SCREENING_DIR,
        "top5_quarterly_stability.csv"
    )
)

q2 = pd.read_csv(
    os.path.join(
        SCREENING_DIR,
        "alternative5_quarterly_stability.csv"
    )
)

quarterly = pd.concat(
    [q1, q2],
    ignore_index=True
)

# ------------------------------------------------------------
# 3. Load episode-support results
# ------------------------------------------------------------

e1 = pd.read_csv(
    os.path.join(
        SCREENING_DIR,
        "top5_episode_support.csv"
    )
)

e2 = pd.read_csv(
    os.path.join(
        SCREENING_DIR,
        "alternative5_episode_support.csv"
    )
)

# Standardize column names
e1 = e1.rename(columns={
    "Supported_Positive_Runs": "Positive_Runs",
    "Supported_48h_Blocks": "Positive_48h_Blocks"
})

episodes = pd.concat(
    [e1, e2],
    ignore_index=True
)

# ------------------------------------------------------------
# 4. Summarize quarterly stability
# ------------------------------------------------------------

quarter_summary = (
    quarterly.groupby("Feature")
    .agg(
        Quarters_With_Condition=(
            "Condition_N",
            lambda x: int((x > 0).sum())
        ),
        Quarters_Lift_Above_1=(
            "Lift",
            lambda x: int((x > 1).sum())
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 5. Load quarter-adjusted comparisons
# ------------------------------------------------------------

adjusted = pd.read_csv(
    os.path.join(
        SCREENING_DIR,
        "top3_quarter_adjusted_summary.csv"
    )
)

adjusted = adjusted[
    ["Feature", "Quarter_Adjusted_RR"]
]

# ------------------------------------------------------------
# 6. Combine all results
# ------------------------------------------------------------

candidate_features = episodes["Feature"].unique()

comparison = ranking.loc[
    ranking["Feature"].isin(candidate_features)
].copy()

comparison = comparison.merge(
    quarter_summary,
    on="Feature",
    how="left"
)

comparison = comparison.merge(
    episodes[
        [
            "Feature",
            "Positive_Runs",
            "Positive_48h_Blocks",
            "Largest_Block_Share"
        ]
    ],
    on="Feature",
    how="left"
)

comparison = comparison.merge(
    adjusted,
    on="Feature",
    how="left"
)

assert len(comparison) == 10
assert comparison["Feature"].is_unique

# ------------------------------------------------------------
# 7. Display comparison
# ------------------------------------------------------------

pd.set_option("display.max_colwidth", None)

print("TOP 10 INVESTIGATED F4Z CANDIDATES")
print("=" * 70)

display(
    comparison[
        [
            "Family",
            "Parent_Sensor",
            "Feature",
            "Condition",
            "Coverage",
            "Event_Probability",
            "Lift",
            "Quarter_Adjusted_RR",
            "Quarters_Lift_Above_1",
            "Quarters_With_Condition",
            "Positive_48h_Blocks",
            "Largest_Block_Share"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 8. Save consolidated comparison
# ------------------------------------------------------------

output_path = os.path.join(
    SCREENING_DIR,
    "top10_investigated_candidate_comparison.csv"
)

comparison.to_csv(output_path, index=False)

print("\nSaved:", output_path)